# Projet 11 — Produisez une étude de marché avec Python

## Préparation et nettoyage des données

Dans le cadre de ce projet, nous accompagnons l’entreprise française **La poule qui chante**, spécialisée dans l’élevage et la vente de poulets sous le label **Agriculture Biologique**.

L’entreprise souhaite évaluer la possibilité de se développer à l’international. Aucun pays ni continent n’étant défini au départ, l’objectif est de construire une base de données pays afin d’identifier, dans un second temps, des groupements de pays pertinents pour une première réflexion stratégique sur l’exportation.

Ce premier notebook est consacré à la préparation et au nettoyage des données issues principalement de la FAO, afin d’obtenir une base exploitable pour l’analyse exploratoire, l’ACP et les méthodes de clustering.


In [1]:
# Importation des librairies nécessaires

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import requests
from zipfile import ZipFile
from io import BytesIO
import re
import html

# Paramétrage de l'affichage
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# Définition des chemins des fichiers

fichier_dispo = Path("DisponibiliteAlimentaire_2017.csv")
fichier_population = Path("Population_2000_2018.csv")

In [3]:
# Chargement des données

dispo = pd.read_csv(fichier_dispo)
population = pd.read_csv(fichier_population)

In [4]:
# Première inspection des données

print("Dimensions de dispo :", dispo.shape)
print("Dimensions de population :", population.shape)

Dimensions de dispo : (176600, 14)
Dimensions de population : (4411, 15)


In [5]:
# Aperçu des premières lignes de la table dispo

dispo.head()

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
0,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5511,Production,2511,Blé et produits,2017,2017,Milliers de tonnes,4281.0,S,Données standardisées
1,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5611,Importations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,2302.0,S,Données standardisées
2,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5072,Variation de stock,2511,Blé et produits,2017,2017,Milliers de tonnes,-119.0,S,Données standardisées
3,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5911,Exportations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,0.0,S,Données standardisées
4,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5301,Disponibilité intérieure,2511,Blé et produits,2017,2017,Milliers de tonnes,6701.0,S,Données standardisées


In [6]:
# Aperçu des premières lignes de la table population

population.head()

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
0,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2000,2000,1000 personnes,20779.953,X,Sources internationales sûres,NaN
1,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2001,2001,1000 personnes,21606.988,X,Sources internationales sûres,NaN
2,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2002,2002,1000 personnes,22600.770,X,Sources internationales sûres,NaN
3,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2003,2003,1000 personnes,23680.871,X,Sources internationales sûres,NaN
4,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2004,2004,1000 personnes,24726.684,X,Sources internationales sûres,NaN


In [7]:
# Liste des colonnes disponibles

print("Colonnes de la table dispo :")
print(dispo.columns)

print("\nColonnes de la table population :")
print(population.columns)

Colonnes de la table dispo :
Index(['Code Domaine', 'Domaine', 'Code zone', 'Zone', 'Code Élément',
       'Élément', 'Code Produit', 'Produit', 'Code année', 'Année', 'Unité',
       'Valeur', 'Symbole', 'Description du Symbole'],
      dtype='object')

Colonnes de la table population :
Index(['Code Domaine', 'Domaine', 'Code zone', 'Zone', 'Code Élément',
       'Élément', 'Code Produit', 'Produit', 'Code année', 'Année', 'Unité',
       'Valeur', 'Symbole', 'Description du Symbole', 'Note'],
      dtype='object')


In [8]:
# Informations générales sur les jeux de données

dispo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176600 entries, 0 to 176599
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Code Domaine            176600 non-null  object 
 1   Domaine                 176600 non-null  object 
 2   Code zone               176600 non-null  int64  
 3   Zone                    176600 non-null  object 
 4   Code Élément            176600 non-null  int64  
 5   Élément                 176600 non-null  object 
 6   Code Produit            176600 non-null  int64  
 7   Produit                 176600 non-null  object 
 8   Code année              176600 non-null  int64  
 9   Année                   176600 non-null  int64  
 10  Unité                   176600 non-null  object 
 11  Valeur                  176600 non-null  float64
 12  Symbole                 176600 non-null  object 
 13  Description du Symbole  176600 non-null  object 
dtypes: float64(1), int64

In [9]:
population.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4411 entries, 0 to 4410
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Code Domaine            4411 non-null   object 
 1   Domaine                 4411 non-null   object 
 2   Code zone               4411 non-null   int64  
 3   Zone                    4411 non-null   object 
 4   Code Élément            4411 non-null   int64  
 5   Élément                 4411 non-null   object 
 6   Code Produit            4411 non-null   int64  
 7   Produit                 4411 non-null   object 
 8   Code année              4411 non-null   int64  
 9   Année                   4411 non-null   int64  
 10  Unité                   4411 non-null   object 
 11  Valeur                  4411 non-null   float64
 12  Symbole                 4411 non-null   object 
 13  Description du Symbole  4411 non-null   object 
 14  Note                    258 non-null    

In [10]:
# Vérification des valeurs manquantes

print("Valeurs manquantes dans la table dispo :")
display(dispo.isna().sum())

print("\nValeurs manquantes dans la table population :")
display(population.isna().sum())

Valeurs manquantes dans la table dispo :


Code Domaine              0
Domaine                   0
Code zone                 0
Zone                      0
Code Élément              0
Élément                   0
Code Produit              0
Produit                   0
Code année                0
Année                     0
Unité                     0
Valeur                    0
Symbole                   0
Description du Symbole    0
dtype: int64


Valeurs manquantes dans la table population :


Code Domaine                 0
Domaine                      0
Code zone                    0
Zone                         0
Code Élément                 0
Élément                      0
Code Produit                 0
Produit                      0
Code année                   0
Année                        0
Unité                        0
Valeur                       0
Symbole                      0
Description du Symbole       0
Note                      4153
dtype: int64

In [11]:
# Liste des produits disponibles dans la table dispo

dispo["Produit"].unique()

array(['Blé et produits', 'Riz et produits', 'Orge et produits',
       'Maïs et produits', 'Seigle et produits', 'Avoine',
       'Millet et produits', 'Sorgho et produits', 'Céréales, Autres',
       'Pommes de Terre et produits', 'Ignames', 'Racines nda',
       'Sucre, canne', 'Sucre, betterave', 'Sucre Eq Brut',
       'Edulcorants Autres', 'Miel', 'Haricots', 'Pois',
       'Légumineuses Autres et produits', 'Noix et produits', 'Soja',
       'Arachides Decortiquees', 'Graines de tournesol',
       'Graines Colza/Moutarde', 'Graines de coton', 'Coco (Incl Coprah)',
       'Sésame', 'Olives', 'Plantes Oleiferes, Autre', 'Huile de Soja',
       "Huile d'Arachide", 'Huile de Tournesol',
       'Huile de Colza&Moutarde', 'Huile Graines de Coton',
       'Huile de Palmistes', 'Huile de Palme', 'Huile de Coco',
       'Huile de Sésame', "Huile d'Olive", 'Huile de Son de Riz',
       'Huile de Germe de Maïs', 'Huil Plantes Oleif Autr',
       'Tomates et produits', 'Oignons', 'Légumes, 

In [12]:
# Sélection des données liées à la viande de volailles

volaille = dispo[dispo["Produit"] == "Viande de Volailles"].copy()

In [13]:
# Vérification de la table filtrée

volaille.shape

(2061, 14)

In [14]:
# Liste des éléments disponibles pour la viande de volailles

volaille["Élément"].unique()

array(['Production', 'Importations - Quantité', 'Variation de stock',
       'Disponibilité intérieure', 'Pertes', 'Résidus', 'Nourriture',
       'Disponibilité alimentaire en quantité (kg/personne/an)',
       'Disponibilité alimentaire (Kcal/personne/jour)',
       'Disponibilité de protéines en quantité (g/personne/jour)',
       'Disponibilité de matière grasse en quantité (g/personne/jour)',
       'Exportations - Quantité', 'Alimentation pour touristes',
       'Traitement', 'Autres utilisations (non alimentaire)',
       'Aliments pour animaux', 'Semences'], dtype=object)

In [15]:
# Nombre de lignes par élément pour la viande de volailles

volaille["Élément"].value_counts()

Élément
Disponibilité alimentaire en quantité (kg/personne/an)           172
Disponibilité de matière grasse en quantité (g/personne/jour)    172
Disponibilité alimentaire (Kcal/personne/jour)                   172
Disponibilité de protéines en quantité (g/personne/jour)         172
Nourriture                                                       170
Importations - Quantité                                          170
Disponibilité intérieure                                         170
Variation de stock                                               169
Production                                                       168
Résidus                                                          164
Exportations - Quantité                                          135
Alimentation pour touristes                                       78
Pertes                                                            67
Traitement                                                        46
Autres utilisations (non a

In [16]:
# Sélection des éléments utiles pour l'analyse


elements_utiles = [
    "Production",
    "Importations - Quantité",
    "Exportations - Quantité",
    "Disponibilité intérieure",
    "Nourriture",
    "Disponibilité alimentaire en quantité (kg/personne/an)",
    "Disponibilité de protéines en quantité (g/personne/jour)",
    "Disponibilité de matière grasse en quantité (g/personne/jour)"]

In [17]:
# Filtrage des données sur les éléments retenus

volaille_selection = volaille[volaille["Élément"].isin(elements_utiles)].copy()

In [18]:
# Vérification de la table filtrée

volaille_selection.shape

(1329, 14)

In [19]:
# Vérification des éléments retenus

volaille_selection["Élément"].value_counts()

Élément
Disponibilité de protéines en quantité (g/personne/jour)         172
Disponibilité de matière grasse en quantité (g/personne/jour)    172
Disponibilité alimentaire en quantité (kg/personne/an)           172
Importations - Quantité                                          170
Disponibilité intérieure                                         170
Nourriture                                                       170
Production                                                       168
Exportations - Quantité                                          135
Name: count, dtype: int64

In [20]:
# Vérification des unités associées aux éléments retenus

volaille_selection.groupby("Élément")["Unité"].unique()

Élément
Disponibilité alimentaire en quantité (kg/personne/an)                           [kg]
Disponibilité de matière grasse en quantité (g/personne/jour)       [g/personne/jour]
Disponibilité de protéines en quantité (g/personne/jour)            [g/personne/jour]
Disponibilité intérieure                                         [Milliers de tonnes]
Exportations - Quantité                                          [Milliers de tonnes]
Importations - Quantité                                          [Milliers de tonnes]
Nourriture                                                       [Milliers de tonnes]
Production                                                       [Milliers de tonnes]
Name: Unité, dtype: object

In [21]:
# Transformation de la table en format pays x variables

base_volaille = volaille_selection.pivot_table(
    index="Zone",
    columns="Élément",
    values="Valeur",
    aggfunc="sum").reset_index()

# Suppression du nom de l'axe des colonnes
base_volaille.columns.name = None

# Aperçu de la base obtenue
base_volaille.head()

,Zone,Disponibilité alimentaire en quantité (kg/personne/an),Disponibilité de matière grasse en quantité (g/personne/jour),Disponibilité de protéines en quantité (g/personne/jour),Disponibilité intérieure,Exportations - Quantité,Importations - Quantité,Nourriture,Production
0,Afghanistan,1.53,0.33,0.54,57.0,NaN,29.0,55.0,28.0
1,Afrique du Sud,35.69,9.25,14.11,2118.0,63.0,514.0,2035.0,1667.0
2,Albanie,16.36,6.45,6.26,47.0,0.0,38.0,47.0,13.0
3,Algérie,6.38,1.50,1.97,277.0,0.0,2.0,264.0,275.0
4,Allemagne,19.47,4.16,7.96,1739.0,646.0,842.0,1609.0,1514.0


In [22]:
# Dimensions de la base volaille

base_volaille.shape

(172, 9)

In [23]:
# Liste des colonnes de la base volaille

base_volaille.columns

Index(['Zone', 'Disponibilité alimentaire en quantité (kg/personne/an)',
       'Disponibilité de matière grasse en quantité (g/personne/jour)',
       'Disponibilité de protéines en quantité (g/personne/jour)',
       'Disponibilité intérieure', 'Exportations - Quantité',
       'Importations - Quantité', 'Nourriture', 'Production'],
      dtype='object')

In [24]:
# Renommage des colonnes pour faciliter l'analyse

base_volaille = base_volaille.rename(columns={
    "Zone": "pays",
    "Disponibilité alimentaire en quantité (kg/personne/an)": "dispo_kg_hab_volaille",
    "Disponibilité de matière grasse en quantité (g/personne/jour)": "dispo_mat_grasse_volaille",
    "Disponibilité de protéines en quantité (g/personne/jour)": "dispo_proteines_volaille",
    "Disponibilité intérieure": "dispo_interieure_volaille",
    "Importations - Quantité": "importations_volaille",
    "Exportations - Quantité": "exportations_volaille",
    "Nourriture": "nourriture_volaille",
    "Production": "production_volaille"})

In [25]:
# Vérification des nouveaux noms de colonnes

base_volaille.head()

,pays,dispo_kg_hab_volaille,dispo_mat_grasse_volaille,dispo_proteines_volaille,dispo_interieure_volaille,exportations_volaille,importations_volaille,nourriture_volaille,production_volaille
0,Afghanistan,1.53,0.33,0.54,57.0,NaN,29.0,55.0,28.0
1,Afrique du Sud,35.69,9.25,14.11,2118.0,63.0,514.0,2035.0,1667.0
2,Albanie,16.36,6.45,6.26,47.0,0.0,38.0,47.0,13.0
3,Algérie,6.38,1.50,1.97,277.0,0.0,2.0,264.0,275.0
4,Allemagne,19.47,4.16,7.96,1739.0,646.0,842.0,1609.0,1514.0


In [26]:
# Vérification des valeurs manquantes après ajout des exportations

base_volaille.isna().sum()

pays                          0
dispo_kg_hab_volaille         0
dispo_mat_grasse_volaille     0
dispo_proteines_volaille      0
dispo_interieure_volaille     2
exportations_volaille        37
importations_volaille         2
nourriture_volaille           2
production_volaille           4
dtype: int64

In [27]:
# Identification des pays avec des valeurs manquantes

base_volaille[base_volaille.isna().any(axis=1)].sort_values("pays")

,pays,dispo_kg_hab_volaille,dispo_mat_grasse_volaille,dispo_proteines_volaille,dispo_interieure_volaille,exportations_volaille,importations_volaille,nourriture_volaille,production_volaille
0,Afghanistan,1.53,0.33,0.54,57.0,NaN,29.0,55.0,28.0
13,Bahamas,43.17,13.33,14.61,26.0,NaN,24.0,16.0,6.0
14,Bangladesh,1.50,0.51,0.47,250.0,NaN,0.0,240.0,249.0
23,Burkina Faso,2.27,0.48,0.77,46.0,NaN,0.0,44.0,46.0
26,Cabo Verde,17.62,3.75,6.52,10.0,NaN,12.0,9.0,1.0
27,Cambodge,2.34,0.85,0.74,38.0,NaN,10.0,37.0,28.0
40,Cuba,23.72,5.70,7.12,342.0,NaN,312.0,269.0,29.0
43,Djibouti,2.68,0.59,0.92,3.0,NaN,3.0,3.0,NaN
54,Gambie,3.53,0.75,1.24,8.0,NaN,16.0,8.0,2.0
56,Grenade,45.70,13.99,15.50,8.0,NaN,7.0,5.0,1.0


In [28]:
# Vérification détaillée des pays présentant des valeurs manquantes sur des variables FAO essentielles

pays_valeurs_manquantes = [
    "Djibouti",
    "Maldives",
    "Ouzbékistan",
    "République démocratique populaire lao"]

volaille_selection[
    volaille_selection["Zone"].isin(pays_valeurs_manquantes)].sort_values(["Zone", "Élément"])

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
46493,FBS,Nouveaux Bilans Alimentaire,72,Djibouti,645,Disponibilité alimentaire en quantité (kg/pers...,2734,Viande de Volailles,2017,2017,kg,2.68,Fc,Donnée calculée
46496,FBS,Nouveaux Bilans Alimentaire,72,Djibouti,684,Disponibilité de matière grasse en quantité (g...,2734,Viande de Volailles,2017,2017,g/personne/jour,0.59,Fc,Donnée calculée
46495,FBS,Nouveaux Bilans Alimentaire,72,Djibouti,674,Disponibilité de protéines en quantité (g/pers...,2734,Viande de Volailles,2017,2017,g/personne/jour,0.92,Fc,Donnée calculée
46490,FBS,Nouveaux Bilans Alimentaire,72,Djibouti,5301,Disponibilité intérieure,2734,Viande de Volailles,2017,2017,Milliers de tonnes,3.00,S,Données standardisées
46488,FBS,Nouveaux Bilans Alimentaire,72,Djibouti,5611,Importations - Quantité,2734,Viande de Volailles,2017,2017,Milliers de tonnes,3.00,S,Données standardisées
46492,FBS,Nouveaux Bilans Alimentaire,72,Djibouti,5142,Nourriture,2734,Viande de Volailles,2017,2017,Milliers de tonnes,3.00,S,Données standardisées
102925,FBS,Nouveaux Bilans Alimentaire,132,Maldives,645,Disponibilité alimentaire en quantité (kg/pers...,2734,Viande de Volailles,2017,2017,kg,13.50,Fc,Donnée calculée
102928,FBS,Nouveaux Bilans Alimentaire,132,Maldives,684,Disponibilité de matière grasse en quantité (g...,2734,Viande de Volailles,2017,2017,g/personne/jour,2.94,Fc,Donnée calculée
102927,FBS,Nouveaux Bilans Alimentaire,132,Maldives,674,Disponibilité de protéines en quantité (g/pers...,2734,Viande de Volailles,2017,2017,g/personne/jour,4.70,Fc,Donnée calculée
102921,FBS,Nouveaux Bilans Alimentaire,132,Maldives,5301,Disponibilité intérieure,2734,Viande de Volailles,2017,2017,Milliers de tonnes,12.00,S,Données standardisées


In [29]:
# Vérification des pays avec des valeurs manquantes

base_volaille[base_volaille.isna().any(axis=1)].sort_values("pays")

,pays,dispo_kg_hab_volaille,dispo_mat_grasse_volaille,dispo_proteines_volaille,dispo_interieure_volaille,exportations_volaille,importations_volaille,nourriture_volaille,production_volaille
0,Afghanistan,1.53,0.33,0.54,57.0,NaN,29.0,55.0,28.0
13,Bahamas,43.17,13.33,14.61,26.0,NaN,24.0,16.0,6.0
14,Bangladesh,1.50,0.51,0.47,250.0,NaN,0.0,240.0,249.0
23,Burkina Faso,2.27,0.48,0.77,46.0,NaN,0.0,44.0,46.0
26,Cabo Verde,17.62,3.75,6.52,10.0,NaN,12.0,9.0,1.0
27,Cambodge,2.34,0.85,0.74,38.0,NaN,10.0,37.0,28.0
40,Cuba,23.72,5.70,7.12,342.0,NaN,312.0,269.0,29.0
43,Djibouti,2.68,0.59,0.92,3.0,NaN,3.0,3.0,NaN
54,Gambie,3.53,0.75,1.24,8.0,NaN,16.0,8.0,2.0
56,Grenade,45.70,13.99,15.50,8.0,NaN,7.0,5.0,1.0


In [30]:
# Vérification des dimensions après traitement

base_volaille.shape

(172, 9)

In [31]:
# Vérification des pays avec des valeurs manquantes après traitement

volaille_selection.groupby("Élément")["Unité"].unique()

Élément
Disponibilité alimentaire en quantité (kg/personne/an)                           [kg]
Disponibilité de matière grasse en quantité (g/personne/jour)       [g/personne/jour]
Disponibilité de protéines en quantité (g/personne/jour)            [g/personne/jour]
Disponibilité intérieure                                         [Milliers de tonnes]
Exportations - Quantité                                          [Milliers de tonnes]
Importations - Quantité                                          [Milliers de tonnes]
Nourriture                                                       [Milliers de tonnes]
Production                                                       [Milliers de tonnes]
Name: Unité, dtype: object

In [32]:
base_volaille.shape

(172, 9)

In [33]:
# Traitement des valeurs manquantes dans la base volaille

# Pour les exportations, l'absence de valeur est interprétée comme une absence d'exportations enregistrées
base_volaille["exportations_volaille"] = base_volaille["exportations_volaille"].fillna(0)

# Pour Djibouti et les Maldives, l'absence de production est interprétée comme une production locale nulle
base_volaille.loc[
    base_volaille["pays"].isin(["Djibouti", "Maldives"]),
    "production_volaille"] = 0

# Suppression des pays avec plusieurs variables de volume manquantes
# Après les traitements spécifiques, les pays présentant encore des valeurs manquantes
# sur les variables essentielles de la base volaille sont supprimés.
base_volaille = base_volaille.dropna().copy()

# Vérification après traitement
base_volaille.isna().sum()

pays                         0
dispo_kg_hab_volaille        0
dispo_mat_grasse_volaille    0
dispo_proteines_volaille     0
dispo_interieure_volaille    0
exportations_volaille        0
importations_volaille        0
nourriture_volaille          0
production_volaille          0
dtype: int64

In [34]:
# Sélection de la population pour l'année 2017

population_2017 = population[population["Année"] == 2017].copy()

population_2017.head()

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
17,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,36296.113,X,Sources internationales sûres,NaN
36,OA,Séries temporelles annuelles,202,Afrique du Sud,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,57009.756,X,Sources internationales sûres,NaN
55,OA,Séries temporelles annuelles,3,Albanie,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,2884.169,X,Sources internationales sûres,NaN
74,OA,Séries temporelles annuelles,4,Algérie,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,41389.189,X,Sources internationales sûres,NaN
93,OA,Séries temporelles annuelles,79,Allemagne,511,Population totale,3010,Population-Estimations,2017,2017,1000 personnes,82658.409,X,Sources internationales sûres,NaN


In [35]:
# Préparation de la population 2017

population_2017 = population_2017[["Zone", "Valeur"]].copy()

population_2017 = population_2017.rename(columns={
    "Zone": "pays",
    "Valeur": "population_2017_milliers"})

# Conversion de la population en nombre de personnes
population_2017["population_2017"] = population_2017["population_2017_milliers"] * 1000

population_2017.head()

,pays,population_2017_milliers,population_2017
17,Afghanistan,36296.113,36296113.0
36,Afrique du Sud,57009.756,57009756.0
55,Albanie,2884.169,2884169.0
74,Algérie,41389.189,41389189.0
93,Allemagne,82658.409,82658409.0


In [36]:
# Vérification des doublons sur la colonne pays

print("Doublons dans base_volaille :", base_volaille["pays"].duplicated().sum())
print("Doublons dans population_2017 :", population_2017["pays"].duplicated().sum())

Doublons dans base_volaille : 0
Doublons dans population_2017 : 0


In [37]:
# Fusion de la base volaille avec la population 2017

base_pays = base_volaille.merge(
    population_2017[["pays", "population_2017"]],
    on="pays",
    how="left")

base_pays.head()

,pays,dispo_kg_hab_volaille,dispo_mat_grasse_volaille,dispo_proteines_volaille,dispo_interieure_volaille,exportations_volaille,importations_volaille,nourriture_volaille,production_volaille,population_2017
0,Afghanistan,1.53,0.33,0.54,57.0,0.0,29.0,55.0,28.0,36296113.0
1,Afrique du Sud,35.69,9.25,14.11,2118.0,63.0,514.0,2035.0,1667.0,57009756.0
2,Albanie,16.36,6.45,6.26,47.0,0.0,38.0,47.0,13.0,2884169.0
3,Algérie,6.38,1.50,1.97,277.0,0.0,2.0,264.0,275.0,41389189.0
4,Allemagne,19.47,4.16,7.96,1739.0,646.0,842.0,1609.0,1514.0,82658409.0


In [38]:
# Vérification des dimensions après fusion

base_pays.shape

(170, 10)

In [39]:
# Vérification des valeurs manquantes après fusion

base_pays.isna().sum()

pays                         0
dispo_kg_hab_volaille        0
dispo_mat_grasse_volaille    0
dispo_proteines_volaille     0
dispo_interieure_volaille    0
exportations_volaille        0
importations_volaille        0
nourriture_volaille          0
production_volaille          0
population_2017              0
dtype: int64

### Vérification de la couverture de l’analyse

Le brief indique qu’il serait idéal de disposer d’au moins **100 pays** dans l’analyse, couvrant au minimum **60 % de la population mondiale**.
Nous vérifions donc que la base construite respecte ces deux critères avant de poursuivre l’analyse.

In [40]:
# Vérification de la couverture de l'analyse

nb_pays = base_pays["pays"].nunique()

population_couverte = base_pays["population_2017"].sum()
population_mondiale = population_2017["population_2017"].sum()

part_population_couverte = population_couverte / population_mondiale * 100

print("Nombre de pays dans l'analyse :", nb_pays)
print("Population couverte :", round(population_couverte))
print("Population mondiale 2017 :", round(population_mondiale))
print("Part de la population mondiale couverte :", round(part_population_couverte, 2), "%")

Nombre de pays dans l'analyse : 170
Population couverte : 7329868983
Population mondiale 2017 : 7548134111
Part de la population mondiale couverte : 97.11 %


Cette première base couvre 170 pays, soit 97,11 % de la population mondiale 2017. Cette couverture pourra évoluer après l’ajout de variables externes issues de l’analyse PESTEL, certaines sources pouvant ne pas couvrir l’ensemble des pays.

In [41]:
# Statistiques descriptives de la base pays

base_pays.describe()

,dispo_kg_hab_volaille,dispo_mat_grasse_volaille,dispo_proteines_volaille,dispo_interieure_volaille,exportations_volaille,importations_volaille,nourriture_volaille,production_volaille,population_2017
count,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,170.000000,1.700000e+02
mean,20.375471,4.937176,7.180235,687.594118,104.970588,89.529412,657.047059,716.658824,4.311688e+07
std,15.875021,4.198588,5.607475,2187.184747,460.628724,186.669983,2136.545796,2487.847959,1.539395e+08
min,0.130000,0.030000,0.040000,2.000000,0.000000,0.000000,2.000000,0.000000,5.204500e+04
25%,6.610000,1.492500,2.237500,30.500000,0.000000,3.000000,28.500000,13.000000,2.855103e+06
50%,18.235000,3.755000,6.585000,100.000000,0.000000,16.000000,99.500000,68.000000,9.757833e+06
75%,30.212500,6.692500,10.415000,368.250000,12.000000,81.250000,365.250000,384.250000,2.971320e+07
max,72.310000,17.860000,27.870000,18266.000000,4223.000000,1069.000000,18100.000000,21914.000000,1.421022e+09


In [42]:
# Pays avec la plus forte disponibilité alimentaire de volaille par habitant

base_pays.sort_values(
    by="dispo_kg_hab_volaille",
    ascending=False).head(10)

,pays,dispo_kg_hab_volaille,dispo_mat_grasse_volaille,dispo_proteines_volaille,dispo_interieure_volaille,exportations_volaille,importations_volaille,nourriture_volaille,production_volaille,population_2017
132,Saint-Vincent-et-les Grenadines,72.31,15.06,25.10,8.0,0.0,9.0,8.0,0.0,109827.0
72,Israël,67.39,12.83,27.87,636.0,3.0,0.0,556.0,629.0,8243848.0
134,Samoa,64.77,13.68,21.88,15.0,0.0,17.0,13.0,0.0,195352.0
133,Sainte-Lucie,56.69,17.86,19.00,11.0,0.0,10.0,10.0,1.0,180954.0
131,Saint-Kitts-et-Nevis,55.77,11.91,19.22,4.0,0.0,4.0,3.0,0.0,52045.0
167,États-Unis d'Amérique,55.68,14.83,19.93,18266.0,3692.0,123.0,18100.0,21914.0,325084756.0
152,Trinité-et-Tobago,54.54,13.05,16.63,76.0,0.0,23.0,75.0,61.0,1384059.0
6,Antigua-et-Barbuda,54.10,17.55,17.77,7.0,0.0,7.0,5.0,0.0,95426.0
31,Chine - RAS de Hong-Kong,53.51,12.70,22.26,280.0,663.0,907.0,391.0,24.0,7306322.0
74,Jamaïque,51.10,12.18,15.71,152.0,1.0,31.0,149.0,128.0,2920848.0


In [43]:
# Pays avec les plus fortes importations de viande de volailles

base_pays.sort_values(
    by="importations_volaille",
    ascending=False).head(10)

,pays,dispo_kg_hab_volaille,dispo_mat_grasse_volaille,dispo_proteines_volaille,dispo_interieure_volaille,exportations_volaille,importations_volaille,nourriture_volaille,production_volaille,population_2017
75,Japon,18.50,3.95,7.24,2415.0,10.0,1069.0,2359.0,2215.0,127502725.0
98,Mexique,32.52,9.27,9.26,4219.0,9.0,972.0,4058.0,3249.0,124777324.0
31,Chine - RAS de Hong-Kong,53.51,12.70,22.26,280.0,663.0,907.0,391.0,24.0,7306322.0
4,Allemagne,19.47,4.16,7.96,1739.0,646.0,842.0,1609.0,1514.0,82658409.0
123,Royaume-Uni de Grande-Bretagne et d'Irlande du...,31.94,6.74,13.77,2234.0,359.0,779.0,2131.0,1814.0,66727461.0
7,Arabie saoudite,43.36,9.38,15.57,1435.0,10.0,722.0,1435.0,616.0,33101179.0
116,Pays-Bas,20.33,3.74,8.48,372.0,1418.0,608.0,346.0,1100.0,17021347.0
1,Afrique du Sud,35.69,9.25,14.11,2118.0,63.0,514.0,2035.0,1667.0,57009756.0
51,France,22.90,6.03,8.95,1573.0,501.0,506.0,1485.0,1750.0,64842509.0
69,Iraq,14.95,3.23,5.37,566.0,0.0,470.0,561.0,96.0,37552781.0


In [44]:
# Pays avec la plus forte production de viande de volailles

base_pays.sort_values(
    by="production_volaille",
    ascending=False).head(10)

,pays,dispo_kg_hab_volaille,dispo_mat_grasse_volaille,dispo_proteines_volaille,dispo_interieure_volaille,exportations_volaille,importations_volaille,nourriture_volaille,production_volaille,population_2017
167,États-Unis d'Amérique,55.68,14.83,19.93,18266.0,3692.0,123.0,18100.0,21914.0,3.250848e+08
34,"Chine, continentale",12.33,4.67,3.96,18161.0,576.0,452.0,17518.0,18236.0,1.421022e+09
21,Brésil,48.03,15.34,15.68,9982.0,4223.0,3.0,9982.0,14201.0,2.078338e+08
52,Fédération de Russie,30.98,6.55,10.44,4556.0,115.0,226.0,4509.0,4444.0,1.455301e+08
66,Inde,2.22,0.47,0.75,3661.0,4.0,0.0,2965.0,3545.0,1.338677e+09
98,Mexique,32.52,9.27,9.26,4219.0,9.0,972.0,4058.0,3249.0,1.247773e+08
118,Pologne,30.30,6.18,12.14,1156.0,1025.0,55.0,1150.0,2351.0,3.795318e+07
67,Indonésie,7.19,1.61,2.42,2323.0,0.0,1.0,1904.0,2301.0,2.646510e+08
75,Japon,18.50,3.95,7.24,2415.0,10.0,1069.0,2359.0,2215.0,1.275027e+08
155,Turquie,20.64,4.50,6.99,1674.0,429.0,3.0,1674.0,2192.0,8.111645e+07


### Création d’un indicateur de dépendance aux importations

Afin d’évaluer l’ouverture des pays aux importations de viande de volailles, nous créons un indicateur mesurant la part des importations dans l’approvisionnement total apparent, défini ici comme la production locale plus les importations.

Cet indicateur permet d’identifier les pays dont l’approvisionnement dépend fortement des importations, ce qui peut représenter une opportunité pour une stratégie d’exportation.

In [45]:
# Création d'un indicateur de dépendance aux importations

base_pays["dependance_importation_volaille"] = (
    base_pays["importations_volaille"] /
    (base_pays["production_volaille"] + base_pays["importations_volaille"]) * 100)

base_pays[[
    "pays",
    "production_volaille",
    "importations_volaille",
    "dependance_importation_volaille"]].head()

,pays,production_volaille,importations_volaille,dependance_importation_volaille
0,Afghanistan,28.0,29.0,50.877193
1,Afrique du Sud,1667.0,514.0,23.567171
2,Albanie,13.0,38.0,74.509804
3,Algérie,275.0,2.0,0.722022
4,Allemagne,1514.0,842.0,35.738540


In [46]:
# Pays les plus dépendants des importations de viande de volailles

base_pays.sort_values(
    by="dependance_importation_volaille",
    ascending=False)[[
    "pays",
    "production_volaille",
    "importations_volaille",
    "dependance_importation_volaille",
    "population_2017"]].head(10)

,pays,production_volaille,importations_volaille,dependance_importation_volaille,population_2017
6,Antigua-et-Barbuda,0.0,7.0,100.0,95426.0
169,Îles Salomon,0.0,6.0,100.0,636039.0
134,Samoa,0.0,17.0,100.0,195352.0
131,Saint-Kitts-et-Nevis,0.0,4.0,100.0,52045.0
132,Saint-Vincent-et-les Grenadines,0.0,9.0,100.0,109827.0
99,Mongolie,0.0,10.0,100.0,3113786.0
87,Luxembourg,0.0,11.0,100.0,591910.0
92,Maldives,0.0,12.0,100.0,496402.0
44,Dominique,0.0,4.0,100.0,71458.0
43,Djibouti,0.0,3.0,100.0,944099.0


Les pays présentant une dépendance de 100 % aux importations correspondent principalement à des pays dont la production locale de viande de volailles est nulle dans la base FAO. Cet indicateur est donc pertinent pour mesurer l’ouverture aux importations, mais il doit être interprété conjointement avec la taille du marché, la population et les autres variables économiques.

In [47]:
# Pays dépendants des importations avec une population supérieure à 5 millions

base_pays[base_pays["population_2017"] >= 5_000_000].sort_values(
    by="dependance_importation_volaille",
    ascending=False)[[
    "pays",
    "production_volaille",
    "importations_volaille",
    "dependance_importation_volaille",
    "population_2017"]].head(10)

,pays,production_volaille,importations_volaille,dependance_importation_volaille,population_2017
31,Chine - RAS de Hong-Kong,24.0,907.0,97.422127,7306322.0
146,Tadjikistan,2.0,38.0,95.000000,8880268.0
37,Congo,7.0,104.0,93.693694,5110695.0
40,Cuba,29.0,312.0,91.495601,11339254.0
63,Haïti,9.0,89.0,90.816327,10982366.0
165,Émirats arabes unis,48.0,433.0,90.020790,9487203.0
25,Bénin,18.0,123.0,87.234043,11175198.0
5,Angola,42.0,277.0,86.833856,29816766.0
69,Iraq,96.0,470.0,83.038869,37552781.0
79,Kirghizistan,7.0,25.0,78.125000,6189733.0


Ce classement met en évidence les pays ou zones dont l’approvisionnement en viande de volailles dépend fortement des importations, tout en excluant les très petits marchés grâce à un seuil de population de 5 millions d’habitants.

Cet indicateur ne suffit toutefois pas à lui seul pour recommander un marché : il devra être croisé avec d’autres variables comme la taille du marché, le pouvoir d’achat, la stabilité politique et les critères issus de l’analyse PESTEL.

In [48]:
# Création d'un indicateur d'importations par habitant

base_pays["importations_kg_hab_volaille"] = (
    base_pays["importations_volaille"] * 1_000_000
    / base_pays["population_2017"])

base_pays[[
    "pays",
    "importations_volaille",
    "population_2017",
    "importations_kg_hab_volaille"]].head()

,pays,importations_volaille,population_2017,importations_kg_hab_volaille
0,Afghanistan,29.0,36296113.0,0.798984
1,Afrique du Sud,514.0,57009756.0,9.016001
2,Albanie,38.0,2884169.0,13.175372
3,Algérie,2.0,41389189.0,0.048322
4,Allemagne,842.0,82658409.0,10.186501


In [49]:
# Arrondi de l'indicateur pour faciliter la lecture

base_pays["importations_kg_hab_volaille"] = base_pays["importations_kg_hab_volaille"].round(2)

In [50]:
# Vérification de l'Albanie

base_pays[base_pays["pays"] == "Albanie"][[
    "pays",
    "production_volaille",
    "importations_volaille",
    "population_2017",
    "dispo_kg_hab_volaille",
    "importations_kg_hab_volaille",
    "dependance_importation_volaille"]]

,pays,production_volaille,importations_volaille,population_2017,dispo_kg_hab_volaille,importations_kg_hab_volaille,dependance_importation_volaille
2,Albanie,13.0,38.0,2884169.0,16.36,13.18,74.509804


### Unités de mesure des variables utilisées

Les variables issues de la FAO ne sont pas toutes exprimées dans les mêmes unités. Certaines variables sont exprimées par habitant, tandis que d’autres correspondent à des volumes totaux en milliers de tonnes.

Il est donc nécessaire de documenter les unités avant de créer des indicateurs dérivés.

### Indicateur de dépendance aux importations

Nous calculons un indicateur de dépendance aux importations afin d'estimer la part des importations dans l'approvisionnement apparent en viande de volailles.

L'approvisionnement apparent est ici défini de manière simplifiée comme :

production locale + importations

L'indicateur est donc calculé ainsi :

importations / (production + importations) × 100

Un niveau proche de 100 % indique que l'approvisionnement repose presque entièrement sur les importations. Un niveau proche de 0 % indique au contraire que le pays s'approvisionne principalement par sa production locale.

### Indicateur d'importations par habitant

Nous calculons également les importations de viande de volailles par habitant. Cet indicateur permet de comparer les pays indépendamment de leur taille démographique.

Les importations étant exprimées en milliers de tonnes, elles sont d'abord converties en kilogrammes :

1 millier de tonnes = 1 000 tonnes = 1 000 000 kg

L'indicateur est donc calculé ainsi :

importations en milliers de tonnes × 1 000 000 / population

In [51]:
# Récapitulatif des unités de mesure

unites_variables = pd.DataFrame({
    "variable": [
        "production_volaille",
        "importations_volaille",
        "exportations_volaille",
        "dispo_interieure_volaille",
        "nourriture_volaille",
        "dispo_kg_hab_volaille",
        "dispo_proteines_volaille",
        "population_2017"
    ],
    "unite": [
        "milliers de tonnes",
        "milliers de tonnes",
        "milliers de tonnes",
        "milliers de tonnes",
        "milliers de tonnes",
        "kg/personne/an",
        "g/personne/jour",
        "personnes"
    ],
    "interpretation": [
        "Production locale de viande de volailles",
        "Quantité importée de viande de volailles",
        "Quantité exportée de viande de volailles",
        "Disponibilité intérieure totale de viande de volailles",
        "Quantité destinée à l'alimentation humaine",
        "Disponibilité annuelle de viande de volailles par habitant",
        "Apport quotidien en protéines lié à la viande de volailles",
        "Population totale du pays en 2017"]})

unites_variables

,variable,unite,interpretation
0,production_volaille,milliers de tonnes,Production locale de viande de volailles
1,importations_volaille,milliers de tonnes,Quantité importée de viande de volailles
2,exportations_volaille,milliers de tonnes,Quantité exportée de viande de volailles
3,dispo_interieure_volaille,milliers de tonnes,Disponibilité intérieure totale de viande de v...
4,nourriture_volaille,milliers de tonnes,Quantité destinée à l'alimentation humaine
5,dispo_kg_hab_volaille,kg/personne/an,Disponibilité annuelle de viande de volailles ...
6,dispo_proteines_volaille,g/personne/jour,Apport quotidien en protéines lié à la viande ...
7,population_2017,personnes,Population totale du pays en 2017


### Récapitulatif des indicateurs calculés

Après avoir documenté les unités de mesure des variables initiales, nous ajoutons plusieurs indicateurs dérivés afin de mieux caractériser l’ouverture des pays aux importations de viande de volailles, le poids relatif de la production locale et le solde import-export.


Les deux indicateurs calculés présentent une forte dispersion entre les pays. La dépendance aux importations varie de 0 % à 100 %, ce qui traduit des situations très différentes entre pays producteurs et pays dépendants des importations.

L’indicateur d’importations par habitant présente également des valeurs élevées pour certains pays ou territoires. Ces valeurs peuvent notamment concerner de petits marchés ou des zones jouant un rôle de hub commercial. Ces indicateurs devront donc être interprétés conjointement avec la population, la production locale et les autres variables de marché.

In [52]:
# Pays avec les plus fortes importations de viande de volailles par habitant

base_pays.sort_values(
    by="importations_kg_hab_volaille",
    ascending=False)[[
    "pays",
    "importations_volaille",
    "population_2017",
    "importations_kg_hab_volaille",
    "dependance_importation_volaille"]].head(10)

,pays,importations_volaille,population_2017,importations_kg_hab_volaille,dependance_importation_volaille
31,Chine - RAS de Hong-Kong,907.0,7306322.0,124.14,97.422127
134,Samoa,17.0,195352.0,87.02,100.000000
132,Saint-Vincent-et-les Grenadines,9.0,109827.0,81.95,100.000000
131,Saint-Kitts-et-Nevis,4.0,52045.0,76.86,100.000000
6,Antigua-et-Barbuda,7.0,95426.0,73.36,100.000000
56,Grenade,7.0,110874.0,63.13,87.500000
13,Bahamas,24.0,381755.0,62.87,80.000000
44,Dominique,4.0,71458.0,55.98,100.000000
133,Sainte-Lucie,10.0,180954.0,55.26,90.909091
119,Polynésie française,15.0,276102.0,54.33,93.750000


Les pays présentant les plus fortes importations de viande de volailles par habitant sont principalement des petits marchés insulaires ou des zones commerciales spécifiques, comme Hong-Kong. Ces valeurs élevées doivent donc être interprétées avec prudence : elles peuvent refléter une forte dépendance aux importations, mais aussi un rôle de hub commercial ou de réexportation.

Cet indicateur reste utile pour comparer l’intensité des importations rapportée à la population, mais il ne suffit pas à lui seul pour prioriser les marchés.

In [53]:
# Pays avec les plus fortes importations par habitant
# parmi les pays de plus de 5 millions d'habitants

base_pays[base_pays["population_2017"] >= 5_000_000].sort_values(
    by="importations_kg_hab_volaille",
    ascending=False)[[
    "pays",
    "importations_volaille",
    "population_2017",
    "importations_kg_hab_volaille",
    "dependance_importation_volaille"]].head(10)

,pays,importations_volaille,population_2017,importations_kg_hab_volaille,dependance_importation_volaille
31,Chine - RAS de Hong-Kong,907.0,7306322.0,124.14,97.422127
165,Émirats arabes unis,433.0,9487203.0,45.64,90.020790
116,Pays-Bas,608.0,17021347.0,35.72,35.597190
16,Belgique,338.0,11419748.0,29.60,42.197253
40,Cuba,312.0,11339254.0,27.52,91.495601
42,Danemark,133.0,5732274.0,23.20,43.464052
7,Arabie saoudite,722.0,33101179.0,21.81,53.961136
37,Congo,104.0,5110695.0,20.35,93.693694
22,Bulgarie,108.0,7102444.0,15.21,50.232558
69,Iraq,470.0,37552781.0,12.52,83.038869


Ce classement filtré sur les pays ou zones de plus de 5 millions d’habitants permet de limiter l’effet des très petits marchés.

Il met en évidence des profils plus exploitables pour une réflexion stratégique, notamment des zones fortement importatrices comme Hong-Kong, les Émirats arabes unis, l’Arabie saoudite ou encore l’Iraq.

Certains pays ou zones, comme Hong-Kong, les Pays-Bas ou la Belgique, doivent toutefois être interprétés avec prudence, car ils peuvent également jouer un rôle de plateforme commerciale ou logistique. Ces indicateurs d’importation ne suffisent donc pas à eux seuls pour recommander un marché, mais ils constituent une première information utile sur l’ouverture des pays aux importations de viande de volailles.


### Création d’un indicateur de production par habitant

Afin de comparer la production locale de viande de volailles entre pays de tailles démographiques différentes, nous calculons un indicateur de production par habitant.

La production étant exprimée en milliers de tonnes, elle est convertie en kilogrammes avant d’être rapportée à la population.

Cet indicateur permet d’identifier les pays dont la production locale est importante relativement à leur population, ce qui peut indiquer une forte capacité de production nationale.

In [54]:
# Création d'un indicateur de production par habitant

base_pays["production_kg_hab_volaille"] = (
    base_pays["production_volaille"] * 1_000_000
    / base_pays["population_2017"])

base_pays["production_kg_hab_volaille"] = base_pays["production_kg_hab_volaille"].round(2)

base_pays[[
    "pays",
    "production_volaille",
    "population_2017",
    "production_kg_hab_volaille"]].head()

,pays,production_volaille,population_2017,production_kg_hab_volaille
0,Afghanistan,28.0,36296113.0,0.77
1,Afrique du Sud,1667.0,57009756.0,29.24
2,Albanie,13.0,2884169.0,4.51
3,Algérie,275.0,41389189.0,6.64
4,Allemagne,1514.0,82658409.0,18.32


In [55]:
# Pays avec la plus forte production de viande de volailles par habitant

base_pays.sort_values(
    by="production_kg_hab_volaille",
    ascending=False)[[
    "pays",
    "production_volaille",
    "population_2017",
    "production_kg_hab_volaille",
    "importations_kg_hab_volaille",
    "dependance_importation_volaille"]].head(10)

,pays,production_volaille,population_2017,production_kg_hab_volaille,importations_kg_hab_volaille,dependance_importation_volaille
72,Israël,629.0,8243848.0,76.30,0.00,0.000000
21,Brésil,14201.0,207833823.0,68.33,0.01,0.021121
167,États-Unis d'Amérique,21914.0,325084756.0,67.41,0.38,0.558152
116,Pays-Bas,1100.0,17021347.0,64.62,35.72,35.597190
118,Pologne,2351.0,37953180.0,61.94,1.45,2.285952
90,Malaisie,1724.0,31104646.0,55.43,2.19,3.794643
17,Belize,20.0,375769.0,53.22,0.00,0.000000
15,Barbade,15.0,286232.0,52.41,6.99,11.764706
10,Australie,1269.0,24584620.0,51.62,0.65,1.245136
65,Hongrie,493.0,9729823.0,50.67,5.96,10.526316


Ce classement met en évidence les pays dont la production locale de viande de volailles est élevée relativement à leur population.

Des pays comme le Brésil, les États-Unis, Israël ou la Pologne présentent une forte production par habitant, ce qui peut traduire une capacité de production nationale importante. Dans une perspective d’exportation, ces pays peuvent donc représenter des marchés plus concurrentiels ou moins dépendants des importations.

Cet indicateur doit toutefois être interprété avec prudence, car certains petits pays peuvent apparaître dans le classement en raison de leur faible population. Il sera donc analysé conjointement avec les indicateurs d’importation, la taille du marché et les variables complémentaires issues de l’analyse PESTEL.


In [56]:
# Récapitulatif des indicateurs calculés

indicateurs_calcules = pd.DataFrame({
    "indicateur": [
        "dependance_importation_volaille",
        "importations_kg_hab_volaille",
        "production_kg_hab_volaille",
        "solde_import_export_volaille"
    ],
    "formule": [
        "importations_volaille / (production_volaille + importations_volaille) * 100",
        "importations_volaille * 1 000 000 / population_2017",
        "production_volaille * 1 000 000 / population_2017",
        "importations_volaille - exportations_volaille"
    ],
    "unite": [
        "%",
        "kg/personne/an",
        "kg/personne/an",
        "milliers de tonnes"
    ],
    "interpretation": [
        "Part des importations dans l'approvisionnement apparent en viande de volailles",
        "Volume annuel de viande de volailles importée rapporté à la population",
        "Volume annuel de viande de volailles produite localement rapporté à la population",
        "Solde net entre importations et exportations de viande de volailles"
    ]
})

indicateurs_calcules


,indicateur,formule,unite,interpretation
0,dependance_importation_volaille,importations_volaille / (production_volaille +...,%,Part des importations dans l'approvisionnement...
1,importations_kg_hab_volaille,importations_volaille * 1 000 000 / population...,kg/personne/an,Volume annuel de viande de volailles importée ...
2,production_kg_hab_volaille,production_volaille * 1 000 000 / population_2017,kg/personne/an,Volume annuel de viande de volailles produite ...
3,solde_import_export_volaille,importations_volaille - exportations_volaille,milliers de tonnes,Solde net entre importations et exportations d...


In [57]:
# Statistiques descriptives des indicateurs calculés

base_pays[[
    "dependance_importation_volaille",
    "importations_kg_hab_volaille",
    "production_kg_hab_volaille"]].describe()

,dependance_importation_volaille,importations_kg_hab_volaille,production_kg_hab_volaille
count,170.000000,170.000000,170.000000
mean,35.807517,11.018882,16.144471
std,35.154214,18.679445,17.297268
min,0.000000,0.000000,0.000000
25%,4.000000,0.470000,2.270000
50%,23.192691,3.970000,9.530000
75%,69.704953,11.820000,25.585000
max,100.000000,124.140000,76.300000


Les indicateurs calculés présentent une forte dispersion entre les pays.

La dépendance aux importations varie de 0 % à 100 %, ce qui permet de distinguer les pays majoritairement approvisionnés par leur production locale de ceux dont l’approvisionnement dépend fortement des importations.

Les importations et la production par habitant présentent également des écarts importants. Ces différences confirment l’intérêt d’utiliser une analyse multivariée afin de comparer les pays selon plusieurs dimensions complémentaires : consommation, production locale, importations et taille du marché.

In [58]:
# Vérification des valeurs infinies dans les indicateurs calculés

indicateurs = [
    "dependance_importation_volaille",
    "importations_kg_hab_volaille",
    "production_kg_hab_volaille"]

np.isinf(base_pays[indicateurs]).sum()

dependance_importation_volaille    0
importations_kg_hab_volaille       0
production_kg_hab_volaille         0
dtype: int64

In [59]:
# Liste des colonnes de la base pays après création des indicateurs

base_pays.columns

Index(['pays', 'dispo_kg_hab_volaille', 'dispo_mat_grasse_volaille',
       'dispo_proteines_volaille', 'dispo_interieure_volaille',
       'exportations_volaille', 'importations_volaille', 'nourriture_volaille',
       'production_volaille', 'population_2017',
       'dependance_importation_volaille', 'importations_kg_hab_volaille',
       'production_kg_hab_volaille'],
      dtype='object')

In [60]:
# Création d'un indicateur de solde import-export

base_pays["solde_import_export_volaille"] = (
    base_pays["importations_volaille"] - base_pays["exportations_volaille"])

# Organisation de la base FAO finale

colonnes_fao_finale = [
    "pays",
    
    # Taille du marché
    "population_2017",
    "dispo_interieure_volaille",
    "nourriture_volaille",
    
    # Consommation / disponibilité par habitant
    "dispo_kg_hab_volaille",
    "dispo_proteines_volaille",
    
    # Production, importations et exportations
    "production_volaille",
    "importations_volaille",
    "exportations_volaille",
    
    # Indicateurs calculés
    "dependance_importation_volaille",
    "importations_kg_hab_volaille",
    "production_kg_hab_volaille",
    "solde_import_export_volaille"]

base_fao_finale = base_pays[colonnes_fao_finale].copy()

base_fao_finale.head()

,pays,population_2017,dispo_interieure_volaille,nourriture_volaille,dispo_kg_hab_volaille,dispo_proteines_volaille,production_volaille,importations_volaille,exportations_volaille,dependance_importation_volaille,importations_kg_hab_volaille,production_kg_hab_volaille,solde_import_export_volaille
0,Afghanistan,36296113.0,57.0,55.0,1.53,0.54,28.0,29.0,0.0,50.877193,0.80,0.77,29.0
1,Afrique du Sud,57009756.0,2118.0,2035.0,35.69,14.11,1667.0,514.0,63.0,23.567171,9.02,29.24,451.0
2,Albanie,2884169.0,47.0,47.0,16.36,6.26,13.0,38.0,0.0,74.509804,13.18,4.51,38.0
3,Algérie,41389189.0,277.0,264.0,6.38,1.97,275.0,2.0,0.0,0.722022,0.05,6.64,2.0
4,Allemagne,82658409.0,1739.0,1609.0,19.47,7.96,1514.0,842.0,646.0,35.738540,10.19,18.32,196.0


In [61]:
# Diagnostic de cohérence de la base FAO

print("base_volaille :", base_volaille.shape)
print("base_pays :", base_pays.shape)
print("base_fao_finale :", base_fao_finale.shape)

print("\nNombre de pays dans base_fao_finale :", base_fao_finale["pays"].nunique())
print("Doublons pays :", base_fao_finale["pays"].duplicated().sum())

print("\nColonnes de base_fao_finale :")
print(base_fao_finale.columns.tolist())

print("\nValeurs manquantes :")
display(base_fao_finale.isna().sum())

base_volaille : (170, 9)
base_pays : (170, 14)
base_fao_finale : (170, 13)

Nombre de pays dans base_fao_finale : 170
Doublons pays : 0

Colonnes de base_fao_finale :
['pays', 'population_2017', 'dispo_interieure_volaille', 'nourriture_volaille', 'dispo_kg_hab_volaille', 'dispo_proteines_volaille', 'production_volaille', 'importations_volaille', 'exportations_volaille', 'dependance_importation_volaille', 'importations_kg_hab_volaille', 'production_kg_hab_volaille', 'solde_import_export_volaille']

Valeurs manquantes :


pays                               0
population_2017                    0
dispo_interieure_volaille          0
nourriture_volaille                0
dispo_kg_hab_volaille              0
dispo_proteines_volaille           0
production_volaille                0
importations_volaille              0
exportations_volaille              0
dependance_importation_volaille    0
importations_kg_hab_volaille       0
production_kg_hab_volaille         0
solde_import_export_volaille       0
dtype: int64

In [62]:
# Validation finale de la base FAO

assert base_fao_finale.shape == (170, 13)
assert base_fao_finale["pays"].nunique() == 170
assert base_fao_finale["pays"].duplicated().sum() == 0
assert base_fao_finale.isna().sum().sum() == 0

print("Base FAO validée : 170 pays, 13 variables, aucun doublon, aucune valeur manquante.")

Base FAO validée : 170 pays, 13 variables, aucun doublon, aucune valeur manquante.


In [63]:
# Export de la base FAO finale

base_fao_finale.to_csv("base_fao_finale.csv", index=False)

In [64]:
# Vérification de l'export

Path("base_fao_finale.csv").exists()

True

## Enrichissement de la base avec des données PESTEL

Les données FAO permettent de caractériser les pays du point de vue du marché de la viande de volailles : consommation, production, importations et disponibilité alimentaire.

Cependant, une décision d’exportation ne dépend pas uniquement du marché alimentaire. Elle dépend aussi de facteurs économiques, politiques, sociaux et logistiques.

Nous allons donc enrichir la base FAO avec des données ouvertes complémentaires, sélectionnées à partir d’une logique PESTEL.

### Choix des indicateurs complémentaires

À partir de la logique PESTEL, nous retenons des indicateurs disponibles en open data afin de compléter la base FAO.

L’objectif est d’ajouter des variables permettant d’évaluer non seulement le marché de la volaille, mais aussi l’attractivité économique, sociale, politique, institutionnelle et logistique des pays.

In [65]:
# Indicateurs complémentaires retenus

indicateurs_pestel = pd.DataFrame({
    "Dimension PESTEL": [
        "Économique",
        "Social",
        "Politique",
        "Institutionnel / légal",
        "Logistique",
        "Commercial / douanier",
        "Institutionnel / affaires",
        "Logistique / importation",
        "Environnemental / bio"
    ],
    "Indicateur": [
        "PIB par habitant, PPA",
        "Population urbaine (% du total)",
        "Stabilité politique",
        "Qualité réglementaire",
        "Indice de performance logistique",
        "Droit de douane moyen appliqué au poulet",
        "Score Doing Business",
        "Score moyen des coûts d'importation",
        "Part des terres arables en agriculture biologique"
    ],
    "Variable finale": [
        "pib_habitant_ppa",
        "population_urbaine_pct",
        "stabilite_politique",
        "qualite_reglementaire",
        "performance_logistique",
        "droit_douane_poulet_moyen_pct",
        "doing_business_score",
        "score_cout_import_moyen",
        "part_terres_arables_bio_pct"
    ],
    "Source": [
        "Banque mondiale",
        "Banque mondiale",
        "Worldwide Governance Indicators",
        "Worldwide Governance Indicators",
        "Banque mondiale - Logistics Performance Index",
        "WITS / UNCTAD TRAINS",
        "Doing Business - Banque mondiale",
        "Doing Business - Banque mondiale",
        "Our World in Data / FAOSTAT"
    ],
    "Intérêt pour l'analyse": [
        "Mesurer le pouvoir d'achat potentiel des consommateurs",
        "Approcher la maturité du marché et des circuits de distribution",
        "Évaluer le risque politique lié à une stratégie d'exportation",
        "Apprécier la qualité du cadre réglementaire",
        "Évaluer la facilité logistique et commerciale d'accès au marché",
        "Mesurer le niveau de protection douanière appliqué au poulet",
        "Approcher la facilité générale à faire des affaires",
        "Évaluer les contraintes opérationnelles liées à l'importation",
        "Approcher la maturité de l'écosystème bio"
    ]
})

indicateurs_pestel


,Dimension PESTEL,Indicateur,Variable finale,Source,Intérêt pour l'analyse
0,Économique,"PIB par habitant, PPA",pib_habitant_ppa,Banque mondiale,Mesurer le pouvoir d'achat potentiel des conso...
1,Social,Population urbaine (% du total),population_urbaine_pct,Banque mondiale,Approcher la maturité du marché et des circuit...
2,Politique,Stabilité politique,stabilite_politique,Worldwide Governance Indicators,Évaluer le risque politique lié à une stratégi...
3,Institutionnel / légal,Qualité réglementaire,qualite_reglementaire,Worldwide Governance Indicators,Apprécier la qualité du cadre réglementaire
4,Logistique,Indice de performance logistique,performance_logistique,Banque mondiale - Logistics Performance Index,Évaluer la facilité logistique et commerciale ...
5,Commercial / douanier,Droit de douane moyen appliqué au poulet,droit_douane_poulet_moyen_pct,WITS / UNCTAD TRAINS,Mesurer le niveau de protection douanière appl...
6,Institutionnel / affaires,Score Doing Business,doing_business_score,Doing Business - Banque mondiale,Approcher la facilité générale à faire des aff...
7,Logistique / importation,Score moyen des coûts d'importation,score_cout_import_moyen,Doing Business - Banque mondiale,Évaluer les contraintes opérationnelles liées ...
8,Environnemental / bio,Part des terres arables en agriculture biologique,part_terres_arables_bio_pct,Our World in Data / FAOSTAT,Approcher la maturité de l'écosystème bio


### Récupération des données complémentaires

Les indicateurs complémentaires retenus proviennent principalement de la Banque mondiale et des Worldwide Governance Indicators.

Afin de conserver une base cohérente avec les données FAO, l’année 2017 sera utilisée lorsque l’information est disponible. Lorsque certaines données ne sont pas disponibles exactement en 2017, l’année disponible la plus proche pourra être utilisée, en le documentant.

In [66]:
# Récupération de la liste des pays et territoires de la Banque mondiale

url_pays = "https://api.worldbank.org/v2/country"

params_pays = {
    "format": "json",
    "per_page": 400}

response_pays = requests.get(url_pays, params=params_pays, timeout=60)
response_pays.raise_for_status()

data_pays = response_pays.json()

pays_banque_mondiale = pd.DataFrame([
    {
        "code_iso3": item["id"],
        "pays_banque_mondiale": item["name"],
        "region_banque_mondiale": item["region"]["value"],
        "niveau_revenu": item["incomeLevel"]["value"]
    }
    for item in data_pays[1]])

pays_banque_mondiale.head()

,code_iso3,pays_banque_mondiale,region_banque_mondiale,niveau_revenu
0,ABW,Aruba,Latin America & Caribbean,High income
1,AFE,Africa Eastern and Southern,Aggregates,Aggregates
2,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
3,AFR,Africa,Aggregates,Aggregates
4,AFW,Africa Western and Central,Aggregates,Aggregates


In [67]:
# Filtrage des pays et territoires réels

pays_banque_mondiale_reels = pays_banque_mondiale[
    pays_banque_mondiale["region_banque_mondiale"] != "Aggregates"
].copy()

print("Nombre total de lignes Banque mondiale :", len(pays_banque_mondiale))
print("Nombre de pays et territoires réels :", len(pays_banque_mondiale_reels))

pays_banque_mondiale_reels.head()

Nombre total de lignes Banque mondiale : 295
Nombre de pays et territoires réels : 217


,code_iso3,pays_banque_mondiale,region_banque_mondiale,niveau_revenu
0,ABW,Aruba,Latin America & Caribbean,High income
2,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
5,AGO,Angola,Sub-Saharan Africa,Lower middle income
6,ALB,Albania,Europe & Central Asia,Upper middle income
7,AND,Andorra,Europe & Central Asia,High income


In [68]:
# Vérification des régions présentes dans la liste Banque mondiale

pays_banque_mondiale["region_banque_mondiale"].value_counts()

region_banque_mondiale
Aggregates                                           78
Europe & Central Asia                                58
Sub-Saharan Africa                                   48
Latin America & Caribbean                            42
East Asia & Pacific                                  37
Middle East, North Africa, Afghanistan & Pakistan    23
South Asia                                            6
North America                                         3
Name: count, dtype: int64

In [69]:
# Fonction de récupération d'un indicateur depuis l'API de la Banque mondiale

def recuperer_indicateur_banque_mondiale(code_indicateur, nom_colonne, annee=2017, source=None):
    """
    Récupère un indicateur depuis l'API de la Banque mondiale
    pour une année donnée, puis conserve uniquement les pays et territoires réels.
    
    La fonction prévoit plusieurs tentatives en cas d'erreur temporaire
    du serveur ou de délai de réponse trop long.
    """
    
    import time
    from requests.exceptions import RequestException
    
    url = f"https://api.worldbank.org/v2/country/all/indicator/{code_indicateur}"
    
    params = {
        "format": "json",
        "per_page": 500,
        "date": annee}
    
    if source is not None:
        params["source"] = source
    
    for tentative in range(5):
        try:
            response = requests.get(url, params=params, timeout=60)
            
            if response.status_code == 200:
                data = response.json()
                
                if len(data) < 2 or data[1] is None:
                    print("Aucune donnée récupérée pour l'indicateur :", code_indicateur)
                    print("Réponse de l'API :", data)
                    return pd.DataFrame()
                
                df = pd.DataFrame([
                    {
                        "code_iso3": item["countryiso3code"],
                        "pays_banque_mondiale": item["country"]["value"],
                        "annee": int(item["date"]),
                        nom_colonne: item["value"]}
                    for item in data[1]])
                
                df = df[
                    df["code_iso3"].isin(pays_banque_mondiale_reels["code_iso3"])                ].copy()
                
                return df
            
            print(f"Tentative {tentative + 1}/5 échouée - statut {response.status_code}")
            time.sleep(5)
        
        except RequestException as erreur:
            print(f"Tentative {tentative + 1}/5 échouée - erreur de connexion ou délai dépassé")
            print(erreur)
            time.sleep(5)
    
    print("Échec de la récupération après plusieurs tentatives :", code_indicateur)
    return pd.DataFrame()

In [70]:
# Récupération du PIB par habitant en PPA

pib_habitant = recuperer_indicateur_banque_mondiale(
    code_indicateur="NY.GDP.PCAP.PP.CD",
    nom_colonne="pib_habitant_ppa",
    annee=2017)

pib_habitant.head()

,code_iso3,pays_banque_mondiale,annee,pib_habitant_ppa
48,AFG,Afghanistan,2017,2335.795862
49,ALB,Albania,2017,14110.683242
50,DZA,Algeria,2017,13493.560749
51,ASM,American Samoa,2017,NaN
52,AND,Andorra,2017,53084.863964


In [71]:
# Vérification de la table PIB par habitant

print("Dimensions :", pib_habitant.shape)

print("\nValeurs manquantes :")
display(pib_habitant.isna().sum())

Dimensions : (217, 4)

Valeurs manquantes :


code_iso3                0
pays_banque_mondiale     0
annee                    0
pib_habitant_ppa        18
dtype: int64

In [72]:
# Pays sans valeur de PIB par habitant en 2017

pib_habitant[
    pib_habitant["pib_habitant_ppa"].isna()
][[
    "code_iso3",
    "pays_banque_mondiale",
    "annee",
    "pib_habitant_ppa"]]

,code_iso3,pays_banque_mondiale,annee,pib_habitant_ppa
51,ASM,American Samoa,2017,NaN
75,VGB,British Virgin Islands,2017,NaN
87,CHI,Channel Islands,2017,NaN
97,CUB,Cuba,2017,NaN
109,ERI,Eritrea,2017,NaN
117,PYF,French Polynesia,2017,NaN
123,GIB,Gibraltar,2017,NaN
127,GUM,Guam,2017,NaN
142,IMN,Isle of Man,2017,NaN
151,PRK,"Korea, Dem. People's Rep.",2017,NaN


Certaines valeurs de PIB par habitant en PPA sont manquantes pour l’année 2017.  
Ces valeurs concernent principalement des territoires ou des pays pour lesquels les données économiques sont incomplètes dans la base de la Banque mondiale.

À ce stade, aucune suppression ni imputation n’est réalisée. Le traitement des valeurs manquantes sera effectué après la fusion avec la base FAO, afin de mesurer leur impact réel sur la base finale d’analyse.

In [73]:
# Récupération de la population urbaine (% du total)

population_urbaine = recuperer_indicateur_banque_mondiale(
    code_indicateur="SP.URB.TOTL.IN.ZS",
    nom_colonne="population_urbaine_pct",
    annee=2017)

population_urbaine.head()

,code_iso3,pays_banque_mondiale,annee,population_urbaine_pct
48,AFG,Afghanistan,2017,24.835282
49,ALB,Albania,2017,55.980276
50,DZA,Algeria,2017,71.322791
51,ASM,American Samoa,2017,83.889039
52,AND,Andorra,2017,88.497704


In [74]:
# Vérification de la table population urbaine

print("Dimensions :", population_urbaine.shape)

print("\nValeurs manquantes :")
display(population_urbaine.isna().sum())

Dimensions : (217, 4)

Valeurs manquantes :


code_iso3                 0
pays_banque_mondiale      0
annee                     0
population_urbaine_pct    0
dtype: int64

### Récupération des indicateurs de gouvernance WGI

Les indicateurs de gouvernance retenus, notamment la stabilité politique et la qualité réglementaire, proviennent des Worldwide Governance Indicators.

Ces indicateurs étant issus d’une base spécifique, ils seront récupérés à partir du fichier CSV complet mis à disposition via DataBank, plutôt que par la même requête API que les indicateurs économiques et sociaux.

In [75]:
# Téléchargement du fichier WGI depuis DataBank

url_wgi = "https://databank.worldbank.org/data/download/WGI_CSV.zip"

response_wgi = requests.get(url_wgi, timeout=60)
response_wgi.raise_for_status()

zip_wgi = ZipFile(BytesIO(response_wgi.content))

# Liste des fichiers contenus dans l'archive
zip_wgi.namelist()

['WGICSV.csv', 'WGICountry.csv', 'WGISeries.csv']

In [76]:
# Chargement du fichier principal des indicateurs WGI

wgi = pd.read_csv(
    zip_wgi.open("WGICSV.csv"),
    na_values=[".."])

wgi.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1996,1998,2000,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Anguilla,AIA,Control of Corruption - Governance estimate (a...,GOV_WGI_CC.EST,NaN,NaN,NaN,NaN,NaN,0.701538,1.327120,1.335303,1.329441,1.330999,1.330945,1.331885,1.329529,1.324633,1.325005,1.164629,1.168519,1.171653,1.169410,1.173224,1.178479,0.555753,0.553182,1.176345,1.173312,1.176234
1,Anguilla,AIA,Control of Corruption - Governance score (0-100),GOV_WGI_CC.SC,NaN,NaN,NaN,NaN,NaN,62.055041,74.717767,74.883415,74.764759,74.796300,74.795189,74.814230,74.766537,74.667434,74.674964,71.428716,71.507439,71.570878,71.525491,71.602690,71.709047,59.104117,59.052076,71.665859,71.604473,71.663617
2,Anguilla,AIA,Control of Corruption - Lower bound of the 90%...,GOV_WGI_CC.SC_LB,NaN,NaN,NaN,NaN,NaN,47.606316,60.269042,60.434690,60.316034,60.347575,60.346463,60.365505,60.317811,60.218709,60.226239,56.979991,57.058714,57.122153,57.076765,57.153965,57.260322,44.655392,44.603351,57.217134,57.155748,57.214892
3,Anguilla,AIA,Control of Corruption - Number of sources,GOV_WGI_CC.SR,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
4,Anguilla,AIA,Control of Corruption - Standard error of the ...,GOV_WGI_CC.SE,NaN,NaN,NaN,NaN,NaN,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253,0.435253


In [77]:
# Liste des indicateurs principaux WGI disponibles

wgi_indicateurs_est = wgi[
    wgi["Indicator Code"].str.endswith(".EST")
][[
    "Indicator Code",
    "Indicator Name"
]].drop_duplicates()

wgi_indicateurs_est

,Indicator Code,Indicator Name
0,GOV_WGI_CC.EST,Control of Corruption - Governance estimate (a...
6,GOV_WGI_GE.EST,Government Effectiveness - Governance estimate...
12,GOV_WGI_PV.EST,Political Stability - Governance estimate (app...
18,GOV_WGI_RQ.EST,Regulatory Quality - Governance estimate (appr...
24,GOV_WGI_RL.EST,Rule of Law - Governance estimate (approx. -2....
30,GOV_WGI_VA.EST,Voice and Accountability - Governance estimate...


In [78]:
# Sélection des indicateurs WGI retenus pour l'année 2017

wgi_2017 = wgi[
    wgi["Indicator Code"].isin([
        "GOV_WGI_PV.EST",
        "GOV_WGI_RQ.EST"])
][[
    "Country Name",
    "Country Code",
    "Indicator Code",
    "2017"]].copy()

wgi_2017.head()

,Country Name,Country Code,Indicator Code,2017
12,Anguilla,AIA,GOV_WGI_PV.EST,1.221763
18,Anguilla,AIA,GOV_WGI_RQ.EST,0.872082
48,Cook Islands,COK,GOV_WGI_PV.EST,NaN
54,Cook Islands,COK,GOV_WGI_RQ.EST,NaN
84,French Guiana,GUF,GOV_WGI_PV.EST,0.313507


In [79]:
# Vérification de la table WGI sélectionnée

wgi_2017.shape

(432, 4)

In [80]:
# Transformation de la table WGI en format pays x indicateurs

wgi_2017_pivot = wgi_2017.pivot_table(
    index=["Country Code", "Country Name"],
    columns="Indicator Code",
    values="2017",
    aggfunc="first"
).reset_index()

wgi_2017_pivot.columns.name = None

wgi_2017_pivot.head()

,Country Code,Country Name,GOV_WGI_PV.EST,GOV_WGI_RQ.EST
0,ABW,Aruba,1.206665,1.266061
1,AFG,Afghanistan,-2.613931,-1.416448
2,AGO,Angola,-0.537214,-1.179988
3,AIA,Anguilla,1.221763,0.872082
4,ALB,Albania,0.163830,0.110524


In [81]:
# Vérification de la table WGI après transformation

wgi_2017_pivot.shape

(212, 4)

In [82]:
# Renommage des colonnes WGI

wgi_2017_pivot = wgi_2017_pivot.rename(columns={
    "Country Code": "code_iso3",
    "Country Name": "pays_banque_mondiale",
    "GOV_WGI_PV.EST": "stabilite_politique",
    "GOV_WGI_RQ.EST": "qualite_reglementaire"})

wgi_2017_pivot.head()

,code_iso3,pays_banque_mondiale,stabilite_politique,qualite_reglementaire
0,ABW,Aruba,1.206665,1.266061
1,AFG,Afghanistan,-2.613931,-1.416448
2,AGO,Angola,-0.537214,-1.179988
3,AIA,Anguilla,1.221763,0.872082
4,ALB,Albania,0.163830,0.110524


In [83]:
# Vérification de la table WGI finale

print("Dimensions :", wgi_2017_pivot.shape)

print("\nValeurs manquantes :")
display(wgi_2017_pivot.isna().sum())

Dimensions : (212, 4)

Valeurs manquantes :


code_iso3                0
pays_banque_mondiale     0
stabilite_politique      0
qualite_reglementaire    0
dtype: int64

### Récupération de l’indice de performance logistique

L’indice de performance logistique permet d’approcher la facilité d’accès logistique et commerciale à un marché.

Cet indicateur n’étant pas nécessairement disponible pour l’année 2017, l’année disponible la plus proche sera utilisée.

In [84]:
# Vérification des années disponibles pour l'indice de performance logistique

url_lpi = "https://api.worldbank.org/v2/country/all/indicator/LP.LPI.OVRL.XQ"

params_lpi = {
    "format": "json",
    "per_page": 20000}

response_lpi = requests.get(url_lpi, params=params_lpi, timeout=60)
response_lpi.raise_for_status()

data_lpi = response_lpi.json()

lpi_historique = pd.DataFrame([
    {
        "code_iso3": item["countryiso3code"],
        "pays_banque_mondiale": item["country"]["value"],
        "annee": int(item["date"]),
        "performance_logistique": item["value"]}
    for item in data_lpi[1]])

lpi_historique = lpi_historique[
    lpi_historique["code_iso3"].isin(pays_banque_mondiale_reels["code_iso3"])].copy()

annees_lpi_disponibles = sorted(
    lpi_historique[lpi_historique["performance_logistique"].notna()]["annee"].unique())

annees_lpi_disponibles

[np.int64(2007),
 np.int64(2010),
 np.int64(2012),
 np.int64(2014),
 np.int64(2016),
 np.int64(2018),
 np.int64(2022)]

L’indice de performance logistique n’est pas disponible pour l’année 2017.  
Après vérification des années disponibles, l’année 2018 est retenue car elle correspond à l’année disponible la plus proche de l’année de référence 2017.

Ce choix permet d’approcher les conditions logistiques autour de la période étudiée.

In [85]:
# Récupération de l'indice de performance logistique

performance_logistique = recuperer_indicateur_banque_mondiale(
    code_indicateur="LP.LPI.OVRL.XQ",
    nom_colonne="performance_logistique",
    annee=2018)

performance_logistique.head()

,code_iso3,pays_banque_mondiale,annee,performance_logistique
48,AFG,Afghanistan,2018,1.95
49,ALB,Albania,2018,2.66
50,DZA,Algeria,2018,2.45
51,ASM,American Samoa,2018,NaN
52,AND,Andorra,2018,NaN


In [86]:
# Vérification de la table performance logistique

print("Dimensions :", performance_logistique.shape)

print("\nValeurs manquantes :")
display(performance_logistique.isna().sum())

Dimensions : (217, 4)

Valeurs manquantes :


code_iso3                  0
pays_banque_mondiale       0
annee                      0
performance_logistique    58
dtype: int64

In [87]:
# Nombre de valeurs disponibles pour la performance logistique

nb_lpi_disponibles = performance_logistique["performance_logistique"].notna().sum()
nb_lpi_manquants = performance_logistique["performance_logistique"].isna().sum()

print("Valeurs disponibles :", nb_lpi_disponibles)
print("Valeurs manquantes :", nb_lpi_manquants)

Valeurs disponibles : 159
Valeurs manquantes : 58


In [88]:
# Pays sans valeur de performance logistique

performance_logistique[
    performance_logistique["performance_logistique"].isna()
][[
    "code_iso3",
    "pays_banque_mondiale",
    "annee",
    "performance_logistique"
]].head(30)

,code_iso3,pays_banque_mondiale,annee,performance_logistique
51,ASM,American Samoa,2018,NaN
52,AND,Andorra,2018,NaN
54,ATG,Antigua and Barbuda,2018,NaN
57,ABW,Aruba,2018,NaN
60,AZE,Azerbaijan,2018,NaN
64,BRB,Barbados,2018,NaN
67,BLZ,Belize,2018,NaN
69,BMU,Bermuda,2018,NaN
73,BWA,Botswana,2018,NaN
75,VGB,British Virgin Islands,2018,NaN


Certaines valeurs de l’indice de performance logistique sont manquantes, même en utilisant l’année 2018, qui est l’année disponible la plus proche de 2017.

L’indicateur couvre 159 pays ou territoires parmi les 217 pays ou territoires réels identifiés dans la base Banque mondiale.

Aucune suppression n’est réalisée à ce stade. Le traitement des valeurs manquantes sera effectué après la fusion avec la base FAO, afin d’évaluer leur impact sur le nombre de pays disponibles pour l’analyse finale.

### Ajout d’indicateurs complémentaires liés au climat des affaires, au coût d’importation et aux droits de douane

En complément des indicateurs déjà retenus, plusieurs dimensions supplémentaires sont intégrées afin de mieux apprécier la faisabilité commerciale d’une stratégie d’exportation.

Le score Doing Business permet d’approcher la facilité générale à faire des affaires dans chaque pays. Il s’agit d’un indicateur synthétique compris entre 0 et 100, où une valeur élevée correspond à un environnement réglementaire plus favorable.

Les coûts d’importation permettent d’évaluer une partie des barrières administratives et logistiques à l’entrée sur un marché. Deux indicateurs sont retenus : le coût de conformité documentaire à l’importation et le coût de conformité aux frontières à l’importation, exprimés en dollars américains. Ces deux coûts permettront ensuite de construire un indicateur synthétique de coût total d’importation.

Cependant, ces coûts ne couvrent pas les droits de douane ni les taxes à l’importation. Or, pour une stratégie d’exportation de viande de volailles, les droits de douane peuvent avoir un impact majeur sur la compétitivité prix et la faisabilité commerciale.

Une variable tarifaire spécifique sera donc recherchée pour approcher les droits de douane appliqués à l’importation de viande de volailles.

In [89]:
# Identification du code HS lié à la viande de volaille à partir de la liste des produits WITS

import xml.etree.ElementTree as ET

url_produits_wits = "https://wits.worldbank.org/API/V1/wits/datasource/trn/product/all"

response_produits_wits = requests.get(url_produits_wits, timeout=60)
response_produits_wits.raise_for_status()

racine_produits = ET.fromstring(response_produits_wits.content)

produits_wits = []

for produit in racine_produits.iter():
    infos_produit = {}
    
    for enfant in produit:
        nom_champ = enfant.tag.split("}")[-1]
        infos_produit[nom_champ] = enfant.text
    
    if infos_produit:
        produits_wits.append(infos_produit)

produits_wits = pd.DataFrame(produits_wits)

mots_cles_volaille = "poultry|fowls|gallus|chicken"

produits_lies_volaille = produits_wits[
    produits_wits.astype(str).apply(
        lambda ligne: ligne.str.contains(mots_cles_volaille, case=False, na=False, regex=True).any(),
        axis=1)].copy()

colonnes_a_afficher = [
    colonne for colonne in produits_lies_volaille.columns
    if any(mot in colonne.lower() for mot in ["code", "product", "description"])]

pd.set_option("display.max_colwidth", 200)

produits_lies_volaille[colonnes_a_afficher].head(30)

,products,product,productdescription
21,NaN,NaN,010511 -- -- Fowls of the species Gallus domesticus
25,NaN,NaN,010515 -- (2012-) -- Guinea fowls
27,NaN,NaN,010591 -- (-1995) Fowls of the species Gallus domesticus
28,NaN,NaN,"010592 -- (1996-2006) -- Fowls of the species Gallus domesticus, weighing not more than 2,000 g"
29,NaN,NaN,"010593 -- (1996-2006) -- Fowls of the species Gallus domesticus, weighing more than 2,000 g"
30,NaN,NaN,010594 -- (2007-) -- Fowls of the species Gallus domesticus
77,NaN,NaN,"020710 -- (-1995) Poultry not cut in pieces, fresh or chilled"
82,NaN,NaN,020721 -- (-1995) Fowls of the species Gallus domesticus
84,NaN,NaN,"020723 -- (-1995) Ducks, geese and guinea fowls"
101,NaN,NaN,"020750 -- (-1995) Poultry livers, frozen"


In [90]:
# Sélection des produits WITS appartenant à la famille HS 0207,
# identifiée comme correspondant aux viandes et abats comestibles de volailles

produits_volaille_wits = produits_wits[
    produits_wits.astype(str).apply(
        lambda ligne: ligne.str.contains("0207", case=False, na=False).any(), axis=1)].copy()

produits_volaille_wits.head(30)

,products,product,productdescription,notes
77,NaN,NaN,"020710 -- (-1995) Poultry not cut in pieces, fresh or chilled",NaN
78,NaN,NaN,"020711 -- (1996-) -- Not cut in pieces, fresh or chilled",NaN
79,NaN,NaN,"020712 -- (1996-) -- Not cut in pieces, frozen",NaN
80,NaN,NaN,"020713 -- (1996-) -- Cuts and offal, fresh or chilled",NaN
81,NaN,NaN,"020714 -- (1996-) -- Cuts and offal, frozen",NaN
82,NaN,NaN,020721 -- (-1995) Fowls of the species Gallus domesticus,NaN
83,NaN,NaN,020722 -- (-1995) Turkeys,NaN
84,NaN,NaN,"020723 -- (-1995) Ducks, geese and guinea fowls",NaN
85,NaN,NaN,"020724 -- (1996-) -- Not cut in pieces, fresh or chilled",NaN
86,NaN,NaN,"020725 -- (1996-) -- Not cut in pieces, frozen",NaN


In [91]:
# Recherche des codes produits HS liés à la viande de volailles dans WITS

import xml.etree.ElementTree as ET

url_produits_wits = "https://wits.worldbank.org/API/V1/wits/datasource/trn/product/all"

response_produits_wits = requests.get(url_produits_wits, timeout=60)
response_produits_wits.raise_for_status()

racine_produits = ET.fromstring(response_produits_wits.content)

produits_wits = []

for produit in racine_produits.iter():
    infos_produit = {}
    
    for enfant in produit:
        nom_champ = enfant.tag.split("}")[-1]
        infos_produit[nom_champ] = enfant.text
    
    if infos_produit:
        produits_wits.append(infos_produit)

produits_wits = pd.DataFrame(produits_wits)

produits_volaille_wits = produits_wits[
    produits_wits.astype(str).apply(
        lambda ligne: ligne.str.contains("0207", case=False, na=False).any(),
        axis=1)].copy()

produits_volaille_wits.head(30)

,products,product,productdescription,notes
77,NaN,NaN,"020710 -- (-1995) Poultry not cut in pieces, fresh or chilled",NaN
78,NaN,NaN,"020711 -- (1996-) -- Not cut in pieces, fresh or chilled",NaN
79,NaN,NaN,"020712 -- (1996-) -- Not cut in pieces, frozen",NaN
80,NaN,NaN,"020713 -- (1996-) -- Cuts and offal, fresh or chilled",NaN
81,NaN,NaN,"020714 -- (1996-) -- Cuts and offal, frozen",NaN
82,NaN,NaN,020721 -- (-1995) Fowls of the species Gallus domesticus,NaN
83,NaN,NaN,020722 -- (-1995) Turkeys,NaN
84,NaN,NaN,"020723 -- (-1995) Ducks, geese and guinea fowls",NaN
85,NaN,NaN,"020724 -- (1996-) -- Not cut in pieces, fresh or chilled",NaN
86,NaN,NaN,"020725 -- (1996-) -- Not cut in pieces, frozen",NaN


In [92]:
# Sélection des codes HS correspondant au poulet

produits_volaille_wits_clean = produits_volaille_wits.copy()

# Extraction du code HS à 6 chiffres depuis la description
produits_volaille_wits_clean["code_hs6"] = (
    produits_volaille_wits_clean["productdescription"]
    .str.extract(r"^(\d{6})"))

# Codes retenus pour la viande de poulet
codes_hs_poulet = ["020711", "020712", "020713", "020714"]

produits_poulet_wits = produits_volaille_wits_clean[
    produits_volaille_wits_clean["code_hs6"].isin(codes_hs_poulet)][[
    "code_hs6",
    "productdescription"]].copy()

# Affichage complet des descriptions produits
with pd.option_context("display.max_colwidth", None):
    display(produits_poulet_wits)

,code_hs6,productdescription
78,020711,"020711 -- (1996-) -- Not cut in pieces, fresh or chilled"
79,020712,"020712 -- (1996-) -- Not cut in pieces, frozen"
80,020713,"020713 -- (1996-) -- Cuts and offal, fresh or chilled"
81,020714,"020714 -- (1996-) -- Cuts and offal, frozen"


La mention « 1996- » indique que ces codes produits sont utilisés dans la nomenclature HS depuis 1996. Elle ne correspond pas à l’année des données tarifaires utilisées dans l’analyse.

In [93]:
# Récupération du code WITS de la France depuis la page officielle de métadonnées pays

import re
import html

url_metadata_pays_wits = "https://wits.worldbank.org/countryprofile/metadata/en/country/all"

response_metadata_wits = requests.get(url_metadata_pays_wits, timeout=60)
response_metadata_wits.raise_for_status()

# Conversion du HTML en texte brut
texte_metadata_wits = html.unescape(response_metadata_wits.text)
texte_metadata_wits = re.sub(r"<[^>]+>", " ", texte_metadata_wits)
texte_metadata_wits = re.sub(r"\s+", " ", texte_metadata_wits)

# Recherche de la ligne France / FRA / 250
correspondance_france = re.search(
    r"\b(France)\s+(FRA)\s+(\d{3})\s+(French Republic)",
    texte_metadata_wits)

if correspondance_france is None:
    raise ValueError("France non trouvée dans la page de métadonnées WITS.")

france_wits = pd.DataFrame([{
    "country_name": correspondance_france.group(1),
    "country_iso3": correspondance_france.group(2),
    "country_code": correspondance_france.group(3),
    "long_name": correspondance_france.group(4)}])

france_wits

,country_name,country_iso3,country_code,long_name
0,France,FRA,250,French Republic


In [94]:
# Définition du pays partenaire à partir de la table WITS

partenaire_exportateur = france_wits.loc[
    france_wits["country_iso3"] == "FRA",
    "country_code"].iloc[0]

partenaire_exportateur

'250'

In [95]:
# Test de récupération d'un droit de douane MFN pour un pays exemple

# Pays importateur test : États-Unis
reporter_test = "840"

# Partenaire 000 = World / tarif MFN général
partenaire_test = "000"

# Produit test : morceaux et abats de poulet, congelés
produit_test = "020714"

annee_tarifaire = 2017
type_donnee_tarifaire = "reported"

url_test_tarif = (
    "https://wits.worldbank.org/API/V1/SDMX/V21/rest/data/"
    f"DF_WITS_Tariff_TRAINS/.{reporter_test}.{partenaire_test}.{produit_test}.{type_donnee_tarifaire}/")

params_test_tarif = {
    "startperiod": annee_tarifaire,
    "endperiod": annee_tarifaire,
    "detail": "Full"}

response_test_tarif = requests.get(
    url_test_tarif,
    params=params_test_tarif,
    timeout=60)

print("Statut de la requête :", response_test_tarif.status_code)
print("URL appelée :", response_test_tarif.url)
print(response_test_tarif.text[:1000])

Statut de la requête : 200
URL appelée : https://wits.worldbank.org/API/V1/SDMX/V21/rest/data/DF_WITS_Tariff_TRAINS/.840.000.020714.reported/?startperiod=2017&endperiod=2017&detail=Full
<?xml version="1.0" encoding="utf-8"?><!--NSI Web Service v5.2.3--><message:GenericData xmlns:footer="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message/footer" xmlns:generic="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/data/generic" xmlns:message="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message" xmlns:common="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xml="http://www.w3.org/XML/1998/namespace"><message:Header><message:ID>DF_WITS_Tariff_TRAINS</message:ID><message:Test>false</message:Test><message:Prepared>2026-07-09T11:47:17</message:Prepared><message:Sender id="WBG_WITS"><common:Name xml:lang="en">World Bank Group - WITS</common:Name><message:Contact><common:Name xml:lang="en">WITS Admin</common:Name><mes

In [96]:
# Extraction de la valeur tarifaire depuis la réponse XML WITS

racine_test_tarif = ET.fromstring(response_test_tarif.content)

def nom_local(element):
    """Supprime le namespace XML pour faciliter la lecture des balises."""
    return element.tag.split("}")[-1]

observations_tarif_test = []

for serie in racine_test_tarif.iter():
    if nom_local(serie) == "Series":
        infos_serie = {}
        
        # Informations de la série : reporter, partner, product, datatype
        for enfant in serie:
            if nom_local(enfant) == "SeriesKey":
                for valeur in enfant:
                    if nom_local(valeur) == "Value":
                        infos_serie[valeur.attrib.get("id")] = valeur.attrib.get("value")
            
            # Observations : année et valeur tarifaire
            if nom_local(enfant) == "Obs":
                observation = infos_serie.copy()
                
                for sous_element in enfant:
                    if nom_local(sous_element) == "ObsDimension":
                        observation["annee"] = sous_element.attrib.get("value")
                    
                    if nom_local(sous_element) == "ObsValue":
                        observation["droit_douane_pct"] = sous_element.attrib.get("value")
                
                observations_tarif_test.append(observation)

tarif_test = pd.DataFrame(observations_tarif_test)

tarif_test

,FREQ,DATATYPE,PRODUCTCODE,PARTNER,REPORTER,annee,droit_douane_pct
0,A,Reported,020714,000,840,2017,0


In [97]:
# Test de récupération d'un droit de douane avec la France comme partenaire

reporter_test = "840"  # États-Unis
partenaire_test = partenaire_exportateur  # France, code WITS 250
produit_test = "020714"  # Morceaux et abats de poulet, congelés

annee_tarifaire = 2017
type_donnee_tarifaire = "reported"

url_test_tarif_france = (
    "https://wits.worldbank.org/API/V1/SDMX/V21/rest/data/"
    f"DF_WITS_Tariff_TRAINS/.{reporter_test}.{partenaire_test}.{produit_test}.{type_donnee_tarifaire}/")

params_test_tarif_france = {
    "startperiod": annee_tarifaire,
    "endperiod": annee_tarifaire,
    "detail": "Full"}

response_test_tarif_france = requests.get(
    url_test_tarif_france,
    params=params_test_tarif_france,
    timeout=60)

print("Statut de la requête :", response_test_tarif_france.status_code)
print("URL appelée :", response_test_tarif_france.url)
print(response_test_tarif_france.text[:1000])

Statut de la requête : 404
URL appelée : https://wits.worldbank.org/API/V1/SDMX/V21/rest/data/DF_WITS_Tariff_TRAINS/.840.250.020714.reported/?startperiod=2017&endperiod=2017&detail=Full
Not Found - NoRecordsFound



Le test avec la France comme partenaire commercial ne retourne pas d’observation dans WITS / UNCTAD TRAINS pour la combinaison testée.

Afin de disposer d’un indicateur tarifaire comparable entre pays, la suite de l’analyse utilisera donc le partenaire « 000 », correspondant au tarif général / MFN appliqué par le pays importateur au produit concerné.

Cette variable ne mesure pas un éventuel accord préférentiel spécifique avec la France ou l’Union européenne, mais elle permet d’approcher le niveau général de protection tarifaire du marché importateur pour la viande de poulet.

In [98]:
# Fonction de récupération d'un droit de douane MFN depuis WITS / UNCTAD TRAINS

def recuperer_droit_douane_wits(reporter, produit, annee=2017, partenaire="000", datatype="reported"):
    """
    Récupère le droit de douane MFN pour un pays importateur, un produit HS6 et une année.
    
    reporter : code numérique WITS du pays importateur
    partenaire : 000 correspond au tarif général / MFN
    produit : code HS6 du produit
    datatype : reported par défaut
    """
    
    url = (
        "https://wits.worldbank.org/API/V1/SDMX/V21/rest/data/"
        f"DF_WITS_Tariff_TRAINS/.{reporter}.{partenaire}.{produit}.{datatype}/")
    
    params = {
        "startperiod": annee,
        "endperiod": annee,
        "detail": "Full"}
    
    response = requests.get(url, params=params, timeout=60)
    
    # Si aucune donnée n'est disponible pour cette combinaison
    if response.status_code == 404:
        return pd.DataFrame()
    
    response.raise_for_status()
    
    racine = ET.fromstring(response.content)
    
    def nom_local(element):
        return element.tag.split("}")[-1]
    
    observations = []
    
    for serie in racine.iter():
        if nom_local(serie) == "Series":
            infos_serie = {}
            
            for enfant in serie:
                if nom_local(enfant) == "SeriesKey":
                    for valeur in enfant:
                        if nom_local(valeur) == "Value":
                            infos_serie[valeur.attrib.get("id")] = valeur.attrib.get("value")
                
                if nom_local(enfant) == "Obs":
                    observation = infos_serie.copy()
                    
                    for sous_element in enfant:
                        if nom_local(sous_element) == "ObsDimension":
                            observation["annee"] = sous_element.attrib.get("value")
                        
                        if nom_local(sous_element) == "ObsValue":
                            observation["droit_douane_pct"] = sous_element.attrib.get("value")
                    
                    observations.append(observation)
    
    df = pd.DataFrame(observations)
    
    if not df.empty:
        df["annee"] = pd.to_numeric(df["annee"])
        df["droit_douane_pct"] = pd.to_numeric(df["droit_douane_pct"])
    
    return df


# Test de la fonction sur les États-Unis et le produit 020714
tarif_test_fonction = recuperer_droit_douane_wits(
    reporter="840",
    produit="020714",
    annee=2017)

tarif_test_fonction

,FREQ,DATATYPE,PRODUCTCODE,PARTNER,REPORTER,annee,droit_douane_pct
0,A,Reported,020714,000,840,2017,0


In [99]:
# Test de récupération des droits de douane pour les quatre codes HS poulet

tarifs_test_poulet = []

for code_hs in codes_hs_poulet:
    tarif_code = recuperer_droit_douane_wits(
        reporter="840",      # États-Unis
        produit=code_hs,
        annee=2017    )
    
    if not tarif_code.empty:
        tarifs_test_poulet.append(tarif_code)

tarifs_test_poulet = pd.concat(tarifs_test_poulet, ignore_index=True)

tarifs_test_poulet

,FREQ,DATATYPE,PRODUCTCODE,PARTNER,REPORTER,annee,droit_douane_pct
0,A,Reported,020711,000,840,2017,0
1,A,Reported,020712,000,840,2017,0
2,A,Reported,020713,000,840,2017,0
3,A,Reported,020714,000,840,2017,0


Afin de ne pas multiplier les variables tarifaires dans l’analyse finale, les quatre codes HS6 correspondant à la viande de poulet seront agrégés en un seul indicateur synthétique : le droit de douane moyen appliqué au poulet.

La variable finale retenue sera donc :

`droit_douane_poulet_moyen_pct`

Elle correspondra à la moyenne des droits de douane observés pour les quatre codes HS6 sélectionnés.

In [100]:
# Récupération des codes pays WITS depuis la page officielle de métadonnées

url_metadata_pays_wits = "https://wits.worldbank.org/countryprofile/metadata/en/country/all"

response_metadata_wits = requests.get(url_metadata_pays_wits, timeout=60)
response_metadata_wits.raise_for_status()

html_metadata_wits = response_metadata_wits.text

# Extraction des lignes du tableau HTML sans utiliser lxml
lignes_html = re.findall(
    r"<tr[^>]*>(.*?)</tr>",
    html_metadata_wits,
    flags=re.S | re.I)

lignes_tableau = []

for ligne in lignes_html:
    cellules = re.findall(
        r"<t[dh][^>]*>(.*?)</t[dh]>",
        ligne,
        flags=re.S | re.I)
    
    cellules = [
        re.sub(
            r"\s+",
            " ",
            html.unescape(re.sub(r"<[^>]+>", " ", cellule))
        ).strip()
        for cellule in cellules]
    
    if cellules:
        lignes_tableau.append(cellules)

# Identification de l'en-tête
entete = [
    ligne for ligne in lignes_tableau
    if "Country Name" in ligne and "Country ISO3" in ligne and "Country Code" in ligne][0]

position_entete = lignes_tableau.index(entete)

# Conservation des lignes ayant le même nombre de colonnes que l'en-tête
donnees_pays = [
    ligne for ligne in lignes_tableau[position_entete + 1:]
    if len(ligne) == len(entete)]

metadata_pays_wits = pd.DataFrame(donnees_pays, columns=entete)

# Préparation de la table de correspondance utile
codes_pays_wits = metadata_pays_wits.rename(columns={
    "Country Name": "pays_wits",
    "Country ISO3": "code_iso3",
    "Country Code": "code_wits",
    "Long Name": "nom_long_wits"
})[[
    "pays_wits",
    "code_iso3",
    "code_wits",
    "nom_long_wits"
]].copy()

codes_pays_wits["code_wits"] = codes_pays_wits["code_wits"].astype(str).str.zfill(3)

# Vérification sur quelques pays
codes_pays_wits[
    codes_pays_wits["code_iso3"].isin(["FRA", "USA", "BRA"])]

,pays_wits,code_iso3,code_wits,nom_long_wits
33,Brazil,BRA,076,Federative Republic of Brazil
87,France,FRA,250,French Republic
249,United States,USA,840,United States of America


In [101]:
# Correspondance entre les pays Banque mondiale et les codes WITS

pays_tarifs_wits = pays_banque_mondiale_reels.merge(
    codes_pays_wits[[
        "code_iso3",
        "code_wits",
        "pays_wits"
    ]],
    on="code_iso3",
    how="left")

print("Nombre de pays Banque mondiale :", len(pays_banque_mondiale_reels))
print("Nombre de pays avec code WITS :", pays_tarifs_wits["code_wits"].notna().sum())
print("Nombre de pays sans code WITS :", pays_tarifs_wits["code_wits"].isna().sum())

pays_tarifs_wits.head()

Nombre de pays Banque mondiale : 217
Nombre de pays avec code WITS : 205
Nombre de pays sans code WITS : 12


,code_iso3,pays_banque_mondiale,region_banque_mondiale,niveau_revenu,code_wits,pays_wits
0,ABW,Aruba,Latin America & Caribbean,High income,533,Aruba
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,004,Afghanistan
2,AGO,Angola,Sub-Saharan Africa,Lower middle income,024,Angola
3,ALB,Albania,Europe & Central Asia,Upper middle income,008,Albania
4,AND,Andorra,Europe & Central Asia,High income,020,Andorra


In [102]:
# Pays Banque mondiale sans correspondance avec un code WITS

pays_tarifs_wits[
    pays_tarifs_wits["code_wits"].isna()][
    [
    "code_iso3",
    "pays_banque_mondiale",
    "region_banque_mondiale",
    "niveau_revenu"]]

,code_iso3,pays_banque_mondiale,region_banque_mondiale,niveau_revenu
34,CHI,Channel Islands,Europe & Central Asia,High income
39,COD,"Congo, Dem. Rep.",Sub-Saharan Africa,Low income
88,IMN,Isle of Man,Europe & Central Asia,High income
112,LIE,Liechtenstein,Europe & Central Asia,High income
119,MAF,St. Martin (French part),Latin America & Caribbean,High income
131,MNE,Montenegro,Europe & Central Asia,Upper middle income
157,PRI,Puerto Rico (US),Latin America & Caribbean,High income
164,ROU,Romania,Europe & Central Asia,High income
176,SRB,Serbia,Europe & Central Asia,Upper middle income
193,TLS,Timor-Leste,East Asia & Pacific,Lower middle income


La correspondance entre la liste des pays de la Banque mondiale et la table de codes WITS permet d’identifier 205 pays ou territoires avec un code WITS disponible, sur 217 pays ou territoires réels.

Douze pays ou territoires ne disposent pas de correspondance directe via le code ISO3. Aucune correction manuelle n’est appliquée à ce stade. L’impact réel sera évalué après la fusion avec la base FAO et les autres indicateurs PESTEL.

In [103]:
# Fonction de calcul du droit de douane moyen pour le poulet

def calculer_droit_douane_poulet_pays(code_wits, code_iso3, pays, annee=2017):
    """
    Calcule le droit de douane moyen appliqué au poulet pour un pays importateur.
    
    Le calcul est réalisé à partir des quatre codes HS6 retenus pour la viande de poulet.
    Le partenaire 000 correspond au tarif général / MFN.
    """
    
    tarifs_codes = []
    
    for code_hs in codes_hs_poulet:
        tarif_code = recuperer_droit_douane_wits(
            reporter=str(code_wits).zfill(3),
            produit=code_hs,
            annee=annee,
            partenaire="000",
            datatype="reported")
        
        if not tarif_code.empty:
            tarifs_codes.append(tarif_code)
    
    if len(tarifs_codes) == 0:
        return {
            "code_iso3": code_iso3,
            "pays_banque_mondiale": pays,
            "code_wits": str(code_wits).zfill(3),
            "annee": annee,
            "nb_codes_hs_disponibles": 0,
            "droit_douane_poulet_moyen_pct": np.nan}
    
    tarifs_codes = pd.concat(tarifs_codes, ignore_index=True)
    
    return {
        "code_iso3": code_iso3,
        "pays_banque_mondiale": pays,
        "code_wits": str(code_wits).zfill(3),
        "annee": annee,
        "nb_codes_hs_disponibles": tarifs_codes["PRODUCTCODE"].nunique(),
        "droit_douane_poulet_moyen_pct": tarifs_codes["droit_douane_pct"].mean()}


# Test sur quelques pays
pays_test_tarifs = pays_tarifs_wits[
    pays_tarifs_wits["code_iso3"].isin(["USA", "BRA", "CAN", "JPN"])].copy()

resultats_test_tarifs = []

for _, ligne in pays_test_tarifs.iterrows():
    resultat = calculer_droit_douane_poulet_pays(
        code_wits=ligne["code_wits"],
        code_iso3=ligne["code_iso3"],
        pays=ligne["pays_banque_mondiale"], annee=2017)
    
    resultats_test_tarifs.append(resultat)

tarifs_poulet_test = pd.DataFrame(resultats_test_tarifs)

tarifs_poulet_test

,code_iso3,pays_banque_mondiale,code_wits,annee,nb_codes_hs_disponibles,droit_douane_poulet_moyen_pct
0,BRA,Brazil,076,2017,4,10.000
1,CAN,Canada,124,2017,4,6.125
2,JPN,Japan,392,2017,4,10.450
3,USA,United States,840,2017,4,0.000


Le test sur quelques pays confirme que l’indicateur tarifaire varie selon les pays importateurs.

Pour chaque pays, les droits de douane disponibles sur les quatre codes HS6 retenus sont agrégés par une moyenne simple. Cette approche permet de conserver une seule variable tarifaire synthétique dans la base finale :

`droit_douane_poulet_moyen_pct`

Cette variable représente le niveau moyen de droit de douane MFN appliqué par le pays importateur à la viande de poulet.

In [104]:
# Calcul ou chargement du droit de douane moyen pour le poulet

fichier_tarifs_poulet = Path("tarifs_poulet_wits.csv")

if fichier_tarifs_poulet.exists():
    # Si le fichier existe déjà, on le recharge pour éviter de refaire toutes les requêtes WITS
    tarifs_poulet_wits = pd.read_csv(
        fichier_tarifs_poulet,
        dtype={"code_wits": str})
    
    print("Fichier tarifs_poulet_wits.csv chargé depuis le disque.")
    print("Dimensions :", tarifs_poulet_wits.shape)

else:
    # Si le fichier n'existe pas encore, on récupère les données depuis WITS
    import time
    
    pays_a_traiter_tarifs = pays_tarifs_wits[
        pays_tarifs_wits["code_wits"].notna()].copy()
    
    resultats_tarifs_poulet = []
    
    total_pays = len(pays_a_traiter_tarifs)
    
    for i, (_, ligne) in enumerate(pays_a_traiter_tarifs.iterrows(), start=1):
        
        try:
            resultat = calculer_droit_douane_poulet_pays(
                code_wits=ligne["code_wits"],
                code_iso3=ligne["code_iso3"],
                pays=ligne["pays_banque_mondiale"],
                annee=2017)
        
        except Exception as erreur:
            resultat = {
                "code_iso3": ligne["code_iso3"],
                "pays_banque_mondiale": ligne["pays_banque_mondiale"],
                "code_wits": ligne["code_wits"],
                "annee": 2017,
                "nb_codes_hs_disponibles": 0,
                "droit_douane_poulet_moyen_pct": np.nan}
            
            print("Erreur pour", ligne["pays_banque_mondiale"], ":", erreur)
        
        resultats_tarifs_poulet.append(resultat)
        
        if i % 20 == 0 or i == total_pays:
            print(f"{i}/{total_pays} pays traités")
        
        time.sleep(0.1)
    
    tarifs_poulet_wits = pd.DataFrame(resultats_tarifs_poulet)
    
    # Sauvegarde pour éviter de refaire les requêtes lors des prochains Run All
    tarifs_poulet_wits.to_csv(fichier_tarifs_poulet, index=False)
    
    print("Fichier tarifs_poulet_wits.csv créé et sauvegardé.")
    print("Dimensions :", tarifs_poulet_wits.shape)

tarifs_poulet_wits.head()

Fichier tarifs_poulet_wits.csv chargé depuis le disque.
Dimensions : (205, 6)


,code_iso3,pays_banque_mondiale,code_wits,annee,nb_codes_hs_disponibles,droit_douane_poulet_moyen_pct
0,ABW,Aruba,533,2017,4,0.0
1,AFG,Afghanistan,004,2017,0,NaN
2,AGO,Angola,024,2017,4,10.0
3,ALB,Albania,008,2017,4,10.0
4,AND,Andorra,020,2017,0,NaN


In [105]:
# Vérification de la table des droits de douane poulet

print("Dimensions :", tarifs_poulet_wits.shape)

print("\nValeurs manquantes :")
display(tarifs_poulet_wits.isna().sum())

print("\nRépartition du nombre de codes HS disponibles :")
display(tarifs_poulet_wits["nb_codes_hs_disponibles"].value_counts().sort_index())

print("\nStatistiques descriptives du droit de douane moyen :")
display(tarifs_poulet_wits["droit_douane_poulet_moyen_pct"].describe())

Dimensions : (205, 6)

Valeurs manquantes :


code_iso3                         0
pays_banque_mondiale              0
code_wits                         0
annee                             0
nb_codes_hs_disponibles           0
droit_douane_poulet_moyen_pct    53
dtype: int64


Répartition du nombre de codes HS disponibles :


nb_codes_hs_disponibles
0     53
4    152
Name: count, dtype: int64


Statistiques descriptives du droit de douane moyen :


count    152.000000
mean      16.490290
std       19.021029
min        0.000000
25%        3.200000
50%       10.000000
75%       25.000000
max       98.125000
Name: droit_douane_poulet_moyen_pct, dtype: float64

La récupération des droits de douane permet d’obtenir une valeur exploitable pour 152 pays sur les 205 pays disposant d’un code WITS.

Pour chaque pays, l’indicateur final correspond à la moyenne simple des droits de douane MFN observés sur les quatre codes HS6 retenus pour représenter la viande de poulet.

La variable finale retenue est :

`droit_douane_poulet_moyen_pct`

Elle permet d’approcher le niveau moyen de protection tarifaire appliqué par chaque pays importateur à la viande de poulet. Les pays sans donnée tarifaire disponible ne sont pas supprimés à ce stade ; l’impact des valeurs manquantes sera évalué après la fusion avec la base FAO et les autres indicateurs PESTEL.

### Récupération du score Doing Business

Le score Doing Business permet d’approcher la facilité générale à faire des affaires dans chaque pays. Il s’agit d’un indicateur synthétique exprimé sur une échelle de 0 à 100, où une valeur élevée correspond à un environnement réglementaire plus favorable.

Cet indicateur est utilisé comme variable complémentaire pour évaluer l’attractivité générale d’un pays dans le cadre d’une stratégie d’exportation.

In [106]:
# Vérification du module nécessaire pour lire les fichiers Excel

import sys
import importlib.util

if importlib.util.find_spec("openpyxl") is None:
    print("Installation du module openpyxl...")
    !{sys.executable} -m pip install openpyxl
else:
    print("Module openpyxl déjà installé.")

Module openpyxl déjà installé.


In [107]:
# Téléchargement du fichier historique Doing Business avec scores

from pathlib import Path

urls_doing_business = [
    "https://archive.doingbusiness.org/content/dam/doingBusiness/excel/db2020/Historical-data---COMPLETE-dataset-with-scores.xlsx",
    "https://archive.doingbusiness.org/content/dam/doingBusiness/excel/db-2021/Historical-Data--DB04-DB20-.xlsx"]

fichier_doing_business_excel = Path("doing_business_historical_scores.xlsx")

if not fichier_doing_business_excel.exists():
    
    fichier_telecharge = False
    
    for url_doing_business_excel in urls_doing_business:
        try:
            response_doing_business_excel = requests.get(
                url_doing_business_excel,
                timeout=120)
            
            if response_doing_business_excel.status_code == 200 and len(response_doing_business_excel.content) > 10000:
                with open(fichier_doing_business_excel, "wb") as fichier:
                    fichier.write(response_doing_business_excel.content)
                
                print("Fichier Doing Business téléchargé depuis :")
                print(url_doing_business_excel)
                
                fichier_telecharge = True
                break
            
            else:
                print("Téléchargement non concluant depuis :")
                print(url_doing_business_excel)
                print("Statut :", response_doing_business_excel.status_code)
        
        except Exception as erreur:
            print("Erreur lors du téléchargement depuis :")
            print(url_doing_business_excel)
            print(erreur)
    
    if not fichier_telecharge:
        raise ValueError("Le fichier Doing Business n'a pas pu être téléchargé.")

else:
    print("Fichier Doing Business déjà présent sur le disque.")

# Vérification des feuilles disponibles dans le fichier Excel
excel_doing_business = pd.ExcelFile(fichier_doing_business_excel)

excel_doing_business.sheet_names

Fichier Doing Business déjà présent sur le disque.


['All Data', 'Methodology', 'Metadata']

In [108]:
# Extraction du score Doing Business pour l'année 2017

doing_business_brut = pd.read_excel(
    fichier_doing_business_excel,
    sheet_name="All Data",
    header=3)

# Recherche automatique de la colonne du score Doing Business
colonnes_score = [
    colonne for colonne in doing_business_brut.columns
    if "Ease of doing business score" in str(colonne)
    and "DB17-20" in str(colonne)]

print("Colonnes de score identifiées :")
display(colonnes_score)

colonne_score_doing_business = colonnes_score[0]

doing_business_score = doing_business_brut[
    doing_business_brut["DB Year"] == 2017
][[
    "Country code",
    "Economy",
    "DB Year",
    colonne_score_doing_business]].copy()

doing_business_score = doing_business_score.rename(columns={
    "Country code": "code_iso3",
    "Economy": "pays_doing_business",
    "DB Year": "annee",
    colonne_score_doing_business: "doing_business_score"})

doing_business_score["doing_business_score"] = pd.to_numeric(
    doing_business_score["doing_business_score"],
    errors="coerce")

doing_business_score = doing_business_score[
    doing_business_score["code_iso3"].notna()].copy()

doing_business_score["annee"] = doing_business_score["annee"].astype(int)

doing_business_score = doing_business_score.reset_index(drop=True)

doing_business_score.to_csv("doing_business_score.csv", index=False)

print("Dimensions :", doing_business_score.shape)

print("\nValeurs manquantes :")
display(doing_business_score.isna().sum())

doing_business_score.head()

Colonnes de score identifiées :


['Ease of doing business score (DB17-20 methodology)']

Dimensions : (212, 4)

Valeurs manquantes :


code_iso3               0
pays_doing_business     0
annee                   0
doing_business_score    0
dtype: int64

,code_iso3,pays_doing_business,annee,doing_business_score
0,AFG,Afghanistan,2017,38.93563
1,ALB,Albania,2017,64.16093
2,DZA,Algeria,2017,46.10981
3,AGO,Angola,2017,37.65132
4,ATG,Antigua and Barbuda,2017,58.80632


Le score Doing Business a finalement été récupéré à partir du fichier historique officiel du projet Doing Business, disponible dans les archives de la Banque mondiale.

La variable retenue est le score « Ease of doing business score », calculé selon la méthodologie DB17-20. Il s’agit d’un indicateur synthétique exprimé sur une échelle de 0 à 100, où une valeur élevée correspond à un environnement réglementaire plus favorable aux affaires.

Pour l’analyse, l’année 2017 est retenue afin de rester cohérent avec l’année de référence utilisée dans la base FAO.

La variable finale intégrée à l’analyse est :

`doing_business_score`

In [109]:
# Recherche des colonnes liées aux coûts d'importation dans le fichier Doing Business

colonnes_importation = [
    colonne for colonne in doing_business_brut.columns
    if "import" in str(colonne).lower()
    or "border" in str(colonne).lower()
    or "documentary" in str(colonne).lower()]

print("Nombre de colonnes identifiées :", len(colonnes_importation))

for colonne in colonnes_importation:
    print(colonne)

Nombre de colonnes identifiées : 25
Rank-Trading across borders
Score-Trading across borders (DB16-20 methodology)
Score-Trading across borders (DB06-15 methodology)
Time to export: Documentary compliance (hours) (DB16-20 methodology)
Score-Time to export: Documentary compliance (hours) (DB16-20 methodology)
Time to import: Documentary compliance (hours) (DB16-20 methodology)
Score-Time to import: Documentary compliance (hours) (DB16-20 methodology)
Time to export: Border compliance (hours) (DB16-20 methodology)
Score-Time to export: Border compliance (hours) (DB16-20 methodology)
Time to import: Border compliance (hours) (DB16-20 methodology)
Score-Time to import: Border compliance (hours) (DB16-20 methodology)
Cost to export: Documentary compliance (USD) (DB16-20 methodology)
Score-Cost to export: Documentary compliance (USD) (DB16-20 methodology)
Cost to import: Documentary compliance (USD) (DB16-20 methodology)
Score-Cost to import: Documentary compliance (USD) (DB16-20 methodology

### Récupération des coûts d’importation

Les coûts d’importation issus du module Doing Business permettent d’approcher les barrières administratives et opérationnelles liées à l’entrée d’un produit sur un marché.

Il ne s’agit pas de droits de douane. Les droits de douane sont exprimés en pourcentage et sont déjà traités séparément avec la variable `droit_douane_poulet_moyen_pct`.

Les indicateurs retenus ici concernent les scores liés aux coûts de conformité documentaire et de conformité aux frontières lors d’une importation. Ces scores sont exprimés sur une échelle de 0 à 100.

Afin de conserver une lecture cohérente avec le score Doing Business, un score moyen de coût d’importation est construit. Une valeur élevée indique une situation plus favorable, c’est-à-dire des coûts d’importation relativement plus faibles.

La variable principale retenue pour l’analyse sera :

`score_cout_import_moyen`


In [110]:
# Extraction des scores liés aux coûts d'importation pour l'année 2017

colonne_score_cout_import_documentaire = "Score-Cost to import: Documentary compliance (USD) (DB16-20 methodology)"
colonne_score_cout_import_frontiere = "Score-Cost to import: Border compliance (USD) (DB16-20 methodology)"

couts_importation = doing_business_brut[
    doing_business_brut["DB Year"] == 2017][[
    "Country code",
    "Economy",
    "DB Year",
    colonne_score_cout_import_documentaire,
    colonne_score_cout_import_frontiere]].copy()

couts_importation = couts_importation.rename(columns={
    "Country code": "code_iso3",
    "Economy": "pays_doing_business",
    "DB Year": "annee",
    colonne_score_cout_import_documentaire: "score_cout_import_documentaire",
    colonne_score_cout_import_frontiere: "score_cout_import_frontiere"})

couts_importation["score_cout_import_documentaire"] = pd.to_numeric(
    couts_importation["score_cout_import_documentaire"], errors="coerce")

couts_importation["score_cout_import_frontiere"] = pd.to_numeric(
    couts_importation["score_cout_import_frontiere"],    errors="coerce")

couts_importation["annee"] = couts_importation["annee"].astype(int)

couts_importation["score_cout_import_moyen"] = couts_importation[[
    "score_cout_import_documentaire",
    "score_cout_import_frontiere"]].mean(axis=1)

couts_importation = couts_importation[
    couts_importation["code_iso3"].notna()].copy()

couts_importation = couts_importation.reset_index(drop=True)

couts_importation.to_csv("couts_importation.csv", index=False)

print("Dimensions :", couts_importation.shape)

print("\nValeurs manquantes :")
display(couts_importation.isna().sum())

couts_importation.head()

Dimensions : (212, 6)

Valeurs manquantes :


code_iso3                         0
pays_doing_business               0
annee                             0
score_cout_import_documentaire    0
score_cout_import_frontiere       0
score_cout_import_moyen           0
dtype: int64

,code_iso3,pays_doing_business,annee,score_cout_import_documentaire,score_cout_import_frontiere,score_cout_import_moyen
0,AFG,Afghanistan,2017,0.00000,37.50000,18.75000
1,ALB,Albania,2017,98.57143,93.55833,96.06488
2,DZA,Algeria,2017,42.88889,65.93519,54.41204
3,AGO,Angola,2017,34.28571,14.16667,24.22619
4,ATG,Antigua and Barbuda,2017,85.71429,54.46759,70.09094


Les coûts d’importation sont intégrés à partir des scores fournis par le fichier Doing Business.

Deux dimensions sont retenues :

- le score lié au coût de conformité documentaire à l’importation ;
- le score lié au coût de conformité aux frontières à l’importation.

Ces scores sont exprimés sur une échelle de 0 à 100. Une valeur élevée correspond à une situation plus favorable, c’est-à-dire à des coûts d’importation relativement plus faibles.

Afin de conserver une seule variable synthétique pour l’analyse, un score moyen est calculé :

`score_cout_import_moyen = moyenne(score_cout_import_documentaire, score_cout_import_frontiere)`

La variable principale retenue pour l’analyse sera donc :

`score_cout_import_moyen`

In [111]:
# Harmonisation des codes ISO3 issus de Doing Business

corrections_codes_doing_business = {
    "ROM": "ROU",  # Roumanie
    "TMP": "TLS"   # Timor-Leste
}

doing_business_score["code_iso3"] = doing_business_score["code_iso3"].replace(
    corrections_codes_doing_business
)

couts_importation["code_iso3"] = couts_importation["code_iso3"].replace(
    corrections_codes_doing_business
)

# Sauvegarde des tables corrigées
doing_business_score.to_csv("doing_business_score.csv", index=False)
couts_importation.to_csv("couts_importation.csv", index=False)

print("Codes Doing Business corrigés et tables sauvegardées.")


Codes Doing Business corrigés et tables sauvegardées.


In [112]:
# Point d'étape des tables complémentaires disponibles

tables_complementaires = {
    "PIB par habitant": pib_habitant,
    "Population urbaine": population_urbaine,
    "WGI": wgi_2017_pivot,
    "Performance logistique": performance_logistique,
    "Droits de douane poulet": tarifs_poulet_wits,
    "Doing Business score": doing_business_score,
    "Scores de coûts d'importation": couts_importation}

for nom_table, table in tables_complementaires.items():
    print("\n" + nom_table)
    print("Dimensions :", table.shape)
    print("Colonnes :", list(table.columns))
    print("Valeurs manquantes totales :", table.isna().sum().sum())


PIB par habitant
Dimensions : (217, 4)
Colonnes : ['code_iso3', 'pays_banque_mondiale', 'annee', 'pib_habitant_ppa']
Valeurs manquantes totales : 18

Population urbaine
Dimensions : (217, 4)
Colonnes : ['code_iso3', 'pays_banque_mondiale', 'annee', 'population_urbaine_pct']
Valeurs manquantes totales : 0

WGI
Dimensions : (212, 4)
Colonnes : ['code_iso3', 'pays_banque_mondiale', 'stabilite_politique', 'qualite_reglementaire']
Valeurs manquantes totales : 0

Performance logistique
Dimensions : (217, 4)
Colonnes : ['code_iso3', 'pays_banque_mondiale', 'annee', 'performance_logistique']
Valeurs manquantes totales : 58

Droits de douane poulet
Dimensions : (205, 6)
Colonnes : ['code_iso3', 'pays_banque_mondiale', 'code_wits', 'annee', 'nb_codes_hs_disponibles', 'droit_douane_poulet_moyen_pct']
Valeurs manquantes totales : 53

Doing Business score
Dimensions : (212, 4)
Colonnes : ['code_iso3', 'pays_doing_business', 'annee', 'doing_business_score']
Valeurs manquantes totales : 0

Scores de

In [113]:
# Consolidation des indicateurs complémentaires par pays

base_pestel = pays_banque_mondiale_reels[[
    "code_iso3",
    "pays_banque_mondiale",
    "region_banque_mondiale",
    "niveau_revenu"]].copy()

base_pestel = base_pestel.merge(
    pib_habitant[[
        "code_iso3",
        "pib_habitant_ppa"]],
    on="code_iso3",
    how="left")

base_pestel = base_pestel.merge(
    population_urbaine[[
        "code_iso3",
        "population_urbaine_pct"]],
    on="code_iso3",
    how="left")

base_pestel = base_pestel.merge(
    wgi_2017_pivot[[
        "code_iso3",
        "stabilite_politique",
        "qualite_reglementaire"]],
    on="code_iso3",
    how="left")

base_pestel = base_pestel.merge(
    performance_logistique[[
        "code_iso3",
        "performance_logistique"]],
    on="code_iso3",
    how="left")

base_pestel = base_pestel.merge(
    tarifs_poulet_wits[[
        "code_iso3",
        "droit_douane_poulet_moyen_pct"]],
    on="code_iso3",
    how="left")

base_pestel = base_pestel.merge(
    doing_business_score[[
        "code_iso3",
        "doing_business_score"]],
    on="code_iso3",
    how="left")

base_pestel = base_pestel.merge(
    couts_importation[[
        "code_iso3",
        "score_cout_import_moyen"]],
    on="code_iso3",
    how="left")

print("Dimensions :", base_pestel.shape)

print("\nValeurs manquantes :")
display(base_pestel.isna().sum())

base_pestel.head()

Dimensions : (217, 12)

Valeurs manquantes :


code_iso3                         0
pays_banque_mondiale              0
region_banque_mondiale            0
niveau_revenu                     0
pib_habitant_ppa                 18
population_urbaine_pct            0
stabilite_politique              12
qualite_reglementaire            12
performance_logistique           58
droit_douane_poulet_moyen_pct    65
doing_business_score             31
score_cout_import_moyen          31
dtype: int64

,code_iso3,pays_banque_mondiale,region_banque_mondiale,niveau_revenu,pib_habitant_ppa,population_urbaine_pct,stabilite_politique,qualite_reglementaire,performance_logistique,droit_douane_poulet_moyen_pct,doing_business_score,score_cout_import_moyen
0,ABW,Aruba,Latin America & Caribbean,High income,37524.914920,62.776849,1.206665,1.266061,NaN,0.0,NaN,NaN
1,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,2335.795862,24.835282,-2.613931,-1.416448,1.95,NaN,38.93563,18.75000
2,AGO,Angola,Sub-Saharan Africa,Lower middle income,8006.836615,65.085053,-0.537214,-1.179988,2.05,10.0,37.65132,24.22619
3,ALB,Albania,Europe & Central Asia,Upper middle income,14110.683242,55.980276,0.163830,0.110524,2.66,10.0,64.16093,96.06488
4,AND,Andorra,Europe & Central Asia,High income,53084.863964,88.497704,1.325091,1.345671,NaN,NaN,NaN,NaN


In [114]:
# Vérification des colonnes disponibles pour faire le lien entre FAO et PESTEL

print("Colonnes de base_fao_finale :")
print(list(base_fao_finale.columns))

print("\nColonnes de base_pays :")
print(list(base_pays.columns))

Colonnes de base_fao_finale :
['pays', 'population_2017', 'dispo_interieure_volaille', 'nourriture_volaille', 'dispo_kg_hab_volaille', 'dispo_proteines_volaille', 'production_volaille', 'importations_volaille', 'exportations_volaille', 'dependance_importation_volaille', 'importations_kg_hab_volaille', 'production_kg_hab_volaille', 'solde_import_export_volaille']

Colonnes de base_pays :
['pays', 'dispo_kg_hab_volaille', 'dispo_mat_grasse_volaille', 'dispo_proteines_volaille', 'dispo_interieure_volaille', 'exportations_volaille', 'importations_volaille', 'nourriture_volaille', 'production_volaille', 'population_2017', 'dependance_importation_volaille', 'importations_kg_hab_volaille', 'production_kg_hab_volaille', 'solde_import_export_volaille']


In [115]:
# Comparaison des noms de pays entre la base FAO et la base PESTEL

print("Exemples de pays dans la base FAO :")
display(base_fao_finale[["pays"]].head(30))

print("\nExemples de pays dans la base PESTEL :")
display(base_pestel[[
    "code_iso3",
    "pays_banque_mondiale"]].head(30))

Exemples de pays dans la base FAO :


,pays
0,Afghanistan
1,Afrique du Sud
2,Albanie
3,Algérie
4,Allemagne
5,Angola
6,Antigua-et-Barbuda
7,Arabie saoudite
8,Argentine
9,Arménie



Exemples de pays dans la base PESTEL :


,code_iso3,pays_banque_mondiale
0,ABW,Aruba
1,AFG,Afghanistan
2,AGO,Angola
3,ALB,Albania
4,AND,Andorra
5,ARE,United Arab Emirates
6,ARG,Argentina
7,ARM,Armenia
8,ASM,American Samoa
9,ATG,Antigua and Barbuda


In [116]:
# Vérification des codes pays disponibles dans les données FAO brutes

codes_pays_fao = dispo[[
    "Code zone",
    "Zone"]].drop_duplicates().sort_values("Zone").reset_index(drop=True)

print("Nombre de pays/zones FAO :", len(codes_pays_fao))

codes_pays_fao.head(40)

Nombre de pays/zones FAO : 174


,Code zone,Zone
0,2,Afghanistan
1,202,Afrique du Sud
2,3,Albanie
3,4,Algérie
4,79,Allemagne
5,7,Angola
6,8,Antigua-et-Barbuda
7,194,Arabie saoudite
8,9,Argentine
9,1,Arménie


In [117]:
# Récupération de la liste des pays Banque mondiale en français

url_pays_fr = "https://api.worldbank.org/v2/fr/country"

params_pays_fr = {
    "format": "json",
    "per_page": 400}

response_pays_fr = requests.get(url_pays_fr, params=params_pays_fr, timeout=60)
response_pays_fr.raise_for_status()

data_pays_fr = response_pays_fr.json()

pays_banque_mondiale_fr = pd.DataFrame([
    {
        "code_iso3": item["id"],
        "pays_banque_mondiale_fr": item["name"],
        "region_banque_mondiale": item["region"]["value"],
        "niveau_revenu": item["incomeLevel"]["value"]}
    for item in data_pays_fr[1]])

pays_banque_mondiale_fr = pays_banque_mondiale_fr[
    pays_banque_mondiale_fr["region_banque_mondiale"] != "Agrégats"].copy()

print("Nombre de pays et territoires réels :", len(pays_banque_mondiale_fr))

pays_banque_mondiale_fr.head(30)

Nombre de pays et territoires réels : 217


,code_iso3,pays_banque_mondiale_fr,region_banque_mondiale,niveau_revenu
0,ABW,Aruba,Amérique latine et Caraïbes,Revenu élevé
2,AFG,Afghanistan,"Moyen-Orient, Afrique du Nord, Afghanistan et Pakistan",Faible revenu
5,AGO,Angola,Afrique subsaharienne,"Revenu intermédiaire, tranche inférieure"
6,ALB,Albanie,Europe et Asie centrale,"Revenu intermédiaire, tranche supérieure"
7,AND,Andorre,Europe et Asie centrale,Revenu élevé
9,ARE,Émirats arabes unis,"Moyen-Orient, Afrique du Nord, Afghanistan et Pakistan",Revenu élevé
10,ARG,Argentine,Amérique latine et Caraïbes,"Revenu intermédiaire, tranche supérieure"
11,ARM,Arménie,Europe et Asie centrale,"Revenu intermédiaire, tranche supérieure"
12,ASM,Samoa américaines,Asie de l’Est et Pacifique,Revenu élevé
13,ATG,Antigua-et-Barbuda,Amérique latine et Caraïbes,Revenu élevé


In [118]:
# Création d'une table de correspondance entre les pays FAO et les codes ISO3 Banque mondiale

import unicodedata
import re

def normaliser_nom_pays(nom):
    """
    Normalise les noms de pays afin de faciliter la correspondance :
    - passage en minuscules
    - suppression des accents
    - suppression du contenu entre parenthèses
    - suppression de certains caractères spéciaux
    """
    
    nom = str(nom).lower()
    
    # Suppression du contenu entre parenthèses
    nom = re.sub(r"\(.*?\)", "", nom)
    
    # Suppression des accents
    nom = unicodedata.normalize("NFKD", nom)
    nom = "".join([caractere for caractere in nom if not unicodedata.combining(caractere)])
    
    # Harmonisation des séparateurs
    nom = nom.replace("-", " ")
    nom = nom.replace("'", " ")
    nom = re.sub(r"[^a-z0-9 ]", " ", nom)
    nom = re.sub(r"\s+", " ", nom).strip()
    
    return nom


# Préparation des noms FAO
correspondance_fao = base_fao_finale[["pays"]].copy()

correspondance_fao["pays_normalise"] = correspondance_fao["pays"].apply(normaliser_nom_pays)

# Préparation des noms Banque mondiale en français
correspondance_bm_fr = pays_banque_mondiale_fr[[
    "code_iso3",
    "pays_banque_mondiale_fr"]].copy()

correspondance_bm_fr["pays_normalise"] = correspondance_bm_fr["pays_banque_mondiale_fr"].apply(normaliser_nom_pays)

# Correspondance automatique
correspondance_fao_iso3 = correspondance_fao.merge(
    correspondance_bm_fr,
    on="pays_normalise",
    how="left")

print("Nombre de pays FAO :", len(correspondance_fao_iso3))
print("Nombre de pays avec code ISO3 trouvé :", correspondance_fao_iso3["code_iso3"].notna().sum())
print("Nombre de pays sans code ISO3 :", correspondance_fao_iso3["code_iso3"].isna().sum())

print("\nPays FAO sans correspondance automatique :")
display(
    correspondance_fao_iso3[
        correspondance_fao_iso3["code_iso3"].isna()][[
        "pays",
        "pays_normalise"]])

Nombre de pays FAO : 170
Nombre de pays avec code ISO3 trouvé : 154
Nombre de pays sans code ISO3 : 16

Pays FAO sans correspondance automatique :


,pays,pays_normalise
32,Chine - RAS de Macao,chine ras de macao
33,"Chine, Taiwan Province de",chine taiwan province de
34,"Chine, continentale",chine continentale
37,Congo,congo
68,Iran (République islamique d'),iran
79,Kirghizistan,kirghizistan
123,Royaume-Uni de Grande-Bretagne et d'Irlande du Nord,royaume uni de grande bretagne et d irlande du nord
126,République de Corée,republique de coree
127,République de Moldova,republique de moldova
129,République populaire démocratique de Corée,republique populaire democratique de coree


In [119]:
# Corrections manuelles des correspondances FAO / ISO3

corrections_iso3 = {
    "chine ras de macao": "MAC",
    "chine taiwan province de": "TWN",
    "chine continentale": "CHN",
    "congo": "COG",
    "iran": "IRN",
    "kirghizistan": "KGZ",
    "royaume uni de grande bretagne et d irlande du nord": "GBR",
    "republique de coree": "KOR",
    "republique de moldova": "MDA",
    "republique populaire democratique de coree": "PRK",
    "republique unie de tanzanie": "TZA",
    "slovaquie": "SVK",
    "tchequie": "CZE",
    "yemen": "YEM",
    "egypte": "EGY",
    "etats unis d amerique": "USA"}

correspondance_fao_iso3["code_iso3"] = correspondance_fao_iso3.apply(
    lambda ligne: corrections_iso3.get(ligne["pays_normalise"], ligne["code_iso3"]),
    axis=1)

print("Nombre de pays FAO :", len(correspondance_fao_iso3))
print("Nombre de pays avec code ISO3 :", correspondance_fao_iso3["code_iso3"].notna().sum())
print("Nombre de pays sans code ISO3 :", correspondance_fao_iso3["code_iso3"].isna().sum())

print("\nPays sans code ISO3 après correction :")
display(
    correspondance_fao_iso3[
        correspondance_fao_iso3["code_iso3"].isna()][[
        "pays",
        "pays_normalise"]])

correspondance_fao_iso3[[
    "pays",
    "code_iso3",
    "pays_banque_mondiale_fr"]].head()

Nombre de pays FAO : 170
Nombre de pays avec code ISO3 : 170
Nombre de pays sans code ISO3 : 0

Pays sans code ISO3 après correction :


,pays,pays_normalise


,pays,code_iso3,pays_banque_mondiale_fr
0,Afghanistan,AFG,Afghanistan
1,Afrique du Sud,ZAF,Afrique du Sud
2,Albanie,ALB,Albanie
3,Algérie,DZA,Algérie
4,Allemagne,DEU,Allemagne


In [120]:
# Documentation des corrections manuelles FAO / ISO3

justification_corrections_iso3 = pd.DataFrame({
    "pays_fao_normalise": list(corrections_iso3.keys()),
    "code_iso3_corrige": list(corrections_iso3.values())})

justification_corrections_iso3["source_correction"] = (
    "Code ISO3 vérifié à partir des référentiels internationaux "
    "ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale")

justification_corrections_iso3

,pays_fao_normalise,code_iso3_corrige,source_correction
0,chine ras de macao,MAC,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale
1,chine taiwan province de,TWN,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale
2,chine continentale,CHN,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale
3,congo,COG,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale
4,iran,IRN,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale
5,kirghizistan,KGZ,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale
6,royaume uni de grande bretagne et d irlande du nord,GBR,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale
7,republique de coree,KOR,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale
8,republique de moldova,MDA,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale
9,republique populaire democratique de coree,PRK,Code ISO3 vérifié à partir des référentiels internationaux ISO 3166-1 alpha-3 / métadonnées pays Banque mondiale


In [121]:
# Ajout du code ISO3 à la base FAO finale

base_fao_iso3 = base_fao_finale.merge(
    correspondance_fao_iso3[[
        "pays",
        "code_iso3"]],
    on="pays",
    how="left")

# Réorganisation des colonnes pour placer code_iso3 au début
colonnes_base_fao_iso3 = [
    "code_iso3",
    "pays"] + [colonne for colonne in base_fao_iso3.columns
    if colonne not in ["code_iso3", "pays"]]

base_fao_iso3 = base_fao_iso3[colonnes_base_fao_iso3]

print("Dimensions :", base_fao_iso3.shape)

print("\nValeurs manquantes :")
display(base_fao_iso3.isna().sum())

print("\nNombre de codes ISO3 uniques :")
print(base_fao_iso3["code_iso3"].nunique())

print("\nNombre de doublons sur code_iso3 :")
print(base_fao_iso3["code_iso3"].duplicated().sum())

base_fao_iso3.head()

Dimensions : (170, 14)

Valeurs manquantes :


code_iso3                          0
pays                               0
population_2017                    0
dispo_interieure_volaille          0
nourriture_volaille                0
dispo_kg_hab_volaille              0
dispo_proteines_volaille           0
production_volaille                0
importations_volaille              0
exportations_volaille              0
dependance_importation_volaille    0
importations_kg_hab_volaille       0
production_kg_hab_volaille         0
solde_import_export_volaille       0
dtype: int64


Nombre de codes ISO3 uniques :
170

Nombre de doublons sur code_iso3 :
0


,code_iso3,pays,population_2017,dispo_interieure_volaille,nourriture_volaille,dispo_kg_hab_volaille,dispo_proteines_volaille,production_volaille,importations_volaille,exportations_volaille,dependance_importation_volaille,importations_kg_hab_volaille,production_kg_hab_volaille,solde_import_export_volaille
0,AFG,Afghanistan,36296113.0,57.0,55.0,1.53,0.54,28.0,29.0,0.0,50.877193,0.80,0.77,29.0
1,ZAF,Afrique du Sud,57009756.0,2118.0,2035.0,35.69,14.11,1667.0,514.0,63.0,23.567171,9.02,29.24,451.0
2,ALB,Albanie,2884169.0,47.0,47.0,16.36,6.26,13.0,38.0,0.0,74.509804,13.18,4.51,38.0
3,DZA,Algérie,41389189.0,277.0,264.0,6.38,1.97,275.0,2.0,0.0,0.722022,0.05,6.64,2.0
4,DEU,Allemagne,82658409.0,1739.0,1609.0,19.47,7.96,1514.0,842.0,646.0,35.738540,10.19,18.32,196.0


In [122]:
# Fusion de la base FAO avec les indicateurs complémentaires PESTEL

base_analyse = base_fao_iso3.merge(
    base_pestel[[
        "code_iso3",
        "region_banque_mondiale",
        "niveau_revenu",
        "pib_habitant_ppa",
        "population_urbaine_pct",
        "stabilite_politique",
        "qualite_reglementaire",
        "performance_logistique",
        "droit_douane_poulet_moyen_pct",
        "doing_business_score",
        "score_cout_import_moyen"]],
    on="code_iso3",
    how="left")

print("Dimensions :", base_analyse.shape)

print("\nNombre de pays :")
print(base_analyse["pays"].nunique())

print("\nNombre de codes ISO3 uniques :")
print(base_analyse["code_iso3"].nunique())

print("\nDoublons sur code_iso3 :")
print(base_analyse["code_iso3"].duplicated().sum())

print("\nValeurs manquantes :")
display(base_analyse.isna().sum())

base_analyse.head()

Dimensions : (170, 24)

Nombre de pays :
170

Nombre de codes ISO3 uniques :
170

Doublons sur code_iso3 :
0

Valeurs manquantes :


code_iso3                           0
pays                                0
population_2017                     0
dispo_interieure_volaille           0
nourriture_volaille                 0
dispo_kg_hab_volaille               0
dispo_proteines_volaille            0
production_volaille                 0
importations_volaille               0
exportations_volaille               0
dependance_importation_volaille     0
importations_kg_hab_volaille        0
production_kg_hab_volaille          0
solde_import_export_volaille        0
region_banque_mondiale              1
niveau_revenu                       1
pib_habitant_ppa                    7
population_urbaine_pct              1
stabilite_politique                 3
qualite_reglementaire               3
performance_logistique             27
droit_douane_poulet_moyen_pct      34
doing_business_score                7
score_cout_import_moyen             7
dtype: int64

,code_iso3,pays,population_2017,dispo_interieure_volaille,nourriture_volaille,dispo_kg_hab_volaille,dispo_proteines_volaille,production_volaille,importations_volaille,exportations_volaille,dependance_importation_volaille,importations_kg_hab_volaille,production_kg_hab_volaille,solde_import_export_volaille,region_banque_mondiale,niveau_revenu,pib_habitant_ppa,population_urbaine_pct,stabilite_politique,qualite_reglementaire,performance_logistique,droit_douane_poulet_moyen_pct,doing_business_score,score_cout_import_moyen
0,AFG,Afghanistan,36296113.0,57.0,55.0,1.53,0.54,28.0,29.0,0.0,50.877193,0.80,0.77,29.0,"Middle East, North Africa, Afghanistan & Pakistan",Low income,2335.795862,24.835282,-2.613931,-1.416448,1.95,NaN,38.93563,18.750000
1,ZAF,Afrique du Sud,57009756.0,2118.0,2035.0,35.69,14.11,1667.0,514.0,63.0,23.567171,9.02,29.24,451.0,Sub-Saharan Africa,Upper middle income,13738.438585,63.331108,-0.382832,0.229679,3.38,16.827381,65.40686,66.973215
2,ALB,Albanie,2884169.0,47.0,47.0,16.36,6.26,13.0,38.0,0.0,74.509804,13.18,4.51,38.0,Europe & Central Asia,Upper middle income,14110.683242,55.980276,0.163830,0.110524,2.66,10.000000,64.16093,96.064880
3,DZA,Algérie,41389189.0,277.0,264.0,6.38,1.97,275.0,2.0,0.0,0.722022,0.05,6.64,2.0,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,13493.560749,71.322791,-0.929505,-0.945914,2.45,30.000001,46.10981,54.412040
4,DEU,Allemagne,82658409.0,1739.0,1609.0,19.47,7.96,1514.0,842.0,646.0,35.738540,10.19,18.32,196.0,Europe & Central Asia,High income,54110.253665,80.984254,0.702224,1.694667,4.20,3.200000,79.55430,100.000000


In [123]:
# Analyse des valeurs manquantes par pays dans la base d'analyse

variables_pestel = [
    "region_banque_mondiale",
    "niveau_revenu",
    "pib_habitant_ppa",
    "population_urbaine_pct",
    "stabilite_politique",
    "qualite_reglementaire",
    "performance_logistique",
    "droit_douane_poulet_moyen_pct",
    "doing_business_score",
    "score_cout_import_moyen"]

base_analyse["nb_valeurs_manquantes_pestel"] = base_analyse[variables_pestel].isna().sum(axis=1)

pays_avec_manquants = base_analyse[
    base_analyse["nb_valeurs_manquantes_pestel"] > 0
][[
    "code_iso3",
    "pays",
    "nb_valeurs_manquantes_pestel"
] + variables_pestel].sort_values(
    "nb_valeurs_manquantes_pestel",
    ascending=False)

print("Nombre de pays avec au moins une valeur PESTEL manquante :")
print(len(pays_avec_manquants))

pays_avec_manquants.head(40)

Nombre de pays avec au moins une valeur PESTEL manquante :
52


,code_iso3,pays,nb_valeurs_manquantes_pestel,region_banque_mondiale,niveau_revenu,pib_habitant_ppa,population_urbaine_pct,stabilite_politique,qualite_reglementaire,performance_logistique,droit_douane_poulet_moyen_pct,doing_business_score,score_cout_import_moyen
33,TWN,"Chine, Taiwan Province de",10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,NCL,Nouvelle-Calédonie,7,East Asia & Pacific,High income,NaN,66.968339,NaN,NaN,NaN,NaN,NaN,NaN
119,PYF,Polynésie française,6,East Asia & Pacific,High income,NaN,61.789694,NaN,NaN,NaN,0.000000,NaN,NaN
129,PRK,République populaire démocratique de Corée,5,East Asia & Pacific,Low income,NaN,61.820608,-0.354386,-2.103745,NaN,NaN,NaN,NaN
154,TKM,Turkménistan,3,Europe & Central Asia,Upper middle income,13278.403035,47.033486,-0.228919,-1.788368,2.41,NaN,NaN,NaN
32,MAC,Chine - RAS de Macao,3,East Asia & Pacific,High income,120774.963081,100.000000,1.313536,1.618421,NaN,0.000000,NaN,NaN
40,CUB,Cuba,3,Latin America & Caribbean,Upper middle income,NaN,76.960327,0.572967,-1.140077,2.20,10.000000,NaN,NaN
80,KIR,Kiribati,2,East Asia & Pacific,Lower middle income,2562.065618,52.973470,0.951951,-0.677652,NaN,NaN,46.74575,62.886905
56,GRD,Grenade,2,Latin America & Caribbean,Upper middle income,15041.452571,36.017330,0.913473,-0.048678,NaN,NaN,53.41014,46.428570
15,BRB,Barbade,2,Latin America & Caribbean,High income,18436.805080,59.259270,0.839990,0.449052,NaN,NaN,57.76071,39.285715


In [124]:
# Répartition du nombre de valeurs PESTEL manquantes par pays

repartition_manquants_pestel = (
    base_analyse["nb_valeurs_manquantes_pestel"]
    .value_counts()
    .sort_index()
    .reset_index())

repartition_manquants_pestel.columns = [
    "nb_valeurs_manquantes_pestel",
    "nombre_de_pays"]

repartition_manquants_pestel

,nb_valeurs_manquantes_pestel,nombre_de_pays
0,0,118
1,1,36
2,2,9
3,3,3
4,5,1
5,6,1
6,7,1
7,10,1


In [125]:
# Liste finale des pays avec trop de valeurs PESTEL manquantes

pays_exclus_manquants = base_analyse[
    base_analyse["nb_valeurs_manquantes_pestel"] > 2
][[
    "code_iso3",
    "pays",
    "nb_valeurs_manquantes_pestel",
    "region_banque_mondiale",
    "niveau_revenu",
    "pib_habitant_ppa",
    "population_urbaine_pct",
    "stabilite_politique",
    "qualite_reglementaire",
    "performance_logistique",
    "droit_douane_poulet_moyen_pct",
    "doing_business_score",
    "score_cout_import_moyen"
]].sort_values(
    "nb_valeurs_manquantes_pestel",
    ascending=False)

print("Nombre de pays exclus :", len(pays_exclus_manquants))

pays_exclus_manquants

Nombre de pays exclus : 7


,code_iso3,pays,nb_valeurs_manquantes_pestel,region_banque_mondiale,niveau_revenu,pib_habitant_ppa,population_urbaine_pct,stabilite_politique,qualite_reglementaire,performance_logistique,droit_douane_poulet_moyen_pct,doing_business_score,score_cout_import_moyen
33,TWN,"Chine, Taiwan Province de",10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,NCL,Nouvelle-Calédonie,7,East Asia & Pacific,High income,NaN,66.968339,NaN,NaN,NaN,NaN,NaN,NaN
119,PYF,Polynésie française,6,East Asia & Pacific,High income,NaN,61.789694,NaN,NaN,NaN,0.0,NaN,NaN
129,PRK,République populaire démocratique de Corée,5,East Asia & Pacific,Low income,NaN,61.820608,-0.354386,-2.103745,NaN,NaN,NaN,NaN
32,MAC,Chine - RAS de Macao,3,East Asia & Pacific,High income,120774.963081,100.000000,1.313536,1.618421,NaN,0.0,NaN,NaN
40,CUB,Cuba,3,Latin America & Caribbean,Upper middle income,NaN,76.960327,0.572967,-1.140077,2.20,10.0,NaN,NaN
154,TKM,Turkménistan,3,Europe & Central Asia,Upper middle income,13278.403035,47.033486,-0.228919,-1.788368,2.41,NaN,NaN,NaN


In [126]:
# Création de la base d'analyse filtrée

base_analyse_filtree = base_analyse[
    base_analyse["nb_valeurs_manquantes_pestel"] <= 2].copy()

pays_exclus_manquants = base_analyse[
    base_analyse["nb_valeurs_manquantes_pestel"] > 2].copy()

print("Nombre de pays dans la base initiale :", len(base_analyse))
print("Nombre de pays exclus :", len(pays_exclus_manquants))
print("Nombre de pays conservés :", len(base_analyse_filtree))

print("\nPays exclus :")
display(
    pays_exclus_manquants[[
        "code_iso3",
        "pays",
        "nb_valeurs_manquantes_pestel"
    ]].sort_values(
        "nb_valeurs_manquantes_pestel",
        ascending=False))

print("\nValeurs manquantes restantes dans la base filtrée :")
display(base_analyse_filtree.isna().sum())

Nombre de pays dans la base initiale : 170
Nombre de pays exclus : 7
Nombre de pays conservés : 163

Pays exclus :


,code_iso3,pays,nb_valeurs_manquantes_pestel
33,TWN,"Chine, Taiwan Province de",10
108,NCL,Nouvelle-Calédonie,7
119,PYF,Polynésie française,6
129,PRK,République populaire démocratique de Corée,5
32,MAC,Chine - RAS de Macao,3
40,CUB,Cuba,3
154,TKM,Turkménistan,3



Valeurs manquantes restantes dans la base filtrée :


code_iso3                           0
pays                                0
population_2017                     0
dispo_interieure_volaille           0
nourriture_volaille                 0
dispo_kg_hab_volaille               0
dispo_proteines_volaille            0
production_volaille                 0
importations_volaille               0
exportations_volaille               0
dependance_importation_volaille     0
importations_kg_hab_volaille        0
production_kg_hab_volaille          0
solde_import_export_volaille        0
region_banque_mondiale              0
niveau_revenu                       0
pib_habitant_ppa                    2
population_urbaine_pct              0
stabilite_politique                 0
qualite_reglementaire               0
performance_logistique             22
droit_douane_poulet_moyen_pct      30
doing_business_score                0
score_cout_import_moyen             0
nb_valeurs_manquantes_pestel        0
dtype: int64

In [127]:
# Traitement des valeurs manquantes restantes par imputation médiane

base_analyse_prete = base_analyse_filtree.copy()

# Conservation du nombre initial de valeurs manquantes avant imputation
base_analyse_prete = base_analyse_prete.rename(columns={
    "nb_valeurs_manquantes_pestel": "nb_valeurs_manquantes_pestel_initial"})

variables_a_imputer = [
    "pib_habitant_ppa",
    "performance_logistique",
    "droit_douane_poulet_moyen_pct"]

for variable in variables_a_imputer:
    
    # Création d'un indicateur permettant de tracer les valeurs imputées
    base_analyse_prete[variable + "_imputee"] = base_analyse_prete[variable].isna().astype(int)
    
    # Imputation par médiane des pays comparables : région + niveau de revenu
    base_analyse_prete[variable] = base_analyse_prete.groupby([
        "region_banque_mondiale",
        "niveau_revenu"
    ])[variable].transform(
        lambda serie: serie.fillna(serie.median()))
    
    # Si certaines valeurs restent manquantes, imputation par médiane du niveau de revenu
    base_analyse_prete[variable] = base_analyse_prete.groupby(
        "niveau_revenu"
    )[variable].transform(
        lambda serie: serie.fillna(serie.median()))
    
    # Dernier recours : médiane globale
    base_analyse_prete[variable] = base_analyse_prete[variable].fillna(
        base_analyse_prete[variable].median())

# Synthèse des imputations réalisées
synthese_imputation = pd.DataFrame({
    "variable": variables_a_imputer,
    "nombre_de_valeurs_imputees": [
        base_analyse_prete[variable + "_imputee"].sum()
        for variable in variables_a_imputer]})

print("Dimensions de la base prête :", base_analyse_prete.shape)

print("\nSynthèse des imputations :")
display(synthese_imputation)

print("\nValeurs manquantes restantes :")
display(base_analyse_prete.isna().sum())

base_analyse_prete.head()

Dimensions de la base prête : (163, 28)

Synthèse des imputations :


,variable,nombre_de_valeurs_imputees
0,pib_habitant_ppa,2
1,performance_logistique,22
2,droit_douane_poulet_moyen_pct,30



Valeurs manquantes restantes :


code_iso3                                0
pays                                     0
population_2017                          0
dispo_interieure_volaille                0
nourriture_volaille                      0
dispo_kg_hab_volaille                    0
dispo_proteines_volaille                 0
production_volaille                      0
importations_volaille                    0
exportations_volaille                    0
dependance_importation_volaille          0
importations_kg_hab_volaille             0
production_kg_hab_volaille               0
solde_import_export_volaille             0
region_banque_mondiale                   0
niveau_revenu                            0
pib_habitant_ppa                         0
population_urbaine_pct                   0
stabilite_politique                      0
qualite_reglementaire                    0
performance_logistique                   0
droit_douane_poulet_moyen_pct            0
doing_business_score                     0
score_cout_

,code_iso3,pays,population_2017,dispo_interieure_volaille,nourriture_volaille,dispo_kg_hab_volaille,dispo_proteines_volaille,production_volaille,importations_volaille,exportations_volaille,dependance_importation_volaille,importations_kg_hab_volaille,production_kg_hab_volaille,solde_import_export_volaille,region_banque_mondiale,niveau_revenu,pib_habitant_ppa,population_urbaine_pct,stabilite_politique,qualite_reglementaire,performance_logistique,droit_douane_poulet_moyen_pct,doing_business_score,score_cout_import_moyen,nb_valeurs_manquantes_pestel_initial,pib_habitant_ppa_imputee,performance_logistique_imputee,droit_douane_poulet_moyen_pct_imputee
0,AFG,Afghanistan,36296113.0,57.0,55.0,1.53,0.54,28.0,29.0,0.0,50.877193,0.80,0.77,29.0,"Middle East, North Africa, Afghanistan & Pakistan",Low income,2335.795862,24.835282,-2.613931,-1.416448,1.95,10.000000,38.93563,18.750000,1,0,0,1
1,ZAF,Afrique du Sud,57009756.0,2118.0,2035.0,35.69,14.11,1667.0,514.0,63.0,23.567171,9.02,29.24,451.0,Sub-Saharan Africa,Upper middle income,13738.438585,63.331108,-0.382832,0.229679,3.38,16.827381,65.40686,66.973215,0,0,0,0
2,ALB,Albanie,2884169.0,47.0,47.0,16.36,6.26,13.0,38.0,0.0,74.509804,13.18,4.51,38.0,Europe & Central Asia,Upper middle income,14110.683242,55.980276,0.163830,0.110524,2.66,10.000000,64.16093,96.064880,0,0,0,0
3,DZA,Algérie,41389189.0,277.0,264.0,6.38,1.97,275.0,2.0,0.0,0.722022,0.05,6.64,2.0,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,13493.560749,71.322791,-0.929505,-0.945914,2.45,30.000001,46.10981,54.412040,0,0,0,0
4,DEU,Allemagne,82658409.0,1739.0,1609.0,19.47,7.96,1514.0,842.0,646.0,35.738540,10.19,18.32,196.0,Europe & Central Asia,High income,54110.253665,80.984254,0.702224,1.694667,4.20,3.200000,79.55430,100.000000,0,0,0,0


### Traitement final des valeurs manquantes

Après la fusion des données FAO avec les indicateurs complémentaires PESTEL, certaines valeurs restaient manquantes dans la base d'analyse.

Afin de limiter la perte d'information, les pays présentant au maximum deux valeurs PESTEL manquantes ont été conservés. Les pays présentant plus de deux valeurs manquantes ont été exclus de l'analyse, car leur profil était jugé insuffisamment documenté pour une comparaison fiable.

Les valeurs manquantes restantes ont été imputées selon une approche progressive :

1. imputation par la médiane des pays appartenant à la même région Banque mondiale et au même niveau de revenu ;
2. si nécessaire, imputation par la médiane du même niveau de revenu ;
3. en dernier recours, imputation par la médiane globale de la variable.

Cette méthode permet de conserver un nombre important de pays tout en limitant l'impact des valeurs extrêmes.

Des variables indicatrices ont été créées afin d'identifier les valeurs imputées :

- `pib_habitant_ppa_imputee`
- `performance_logistique_imputee`
- `droit_douane_poulet_moyen_pct_imputee`

La base finale contient 163 pays et ne présente plus aucune valeur manquante.

In [128]:
# Synthèse de la couverture finale de la base d'analyse

nb_pays_fao_initial = len(base_analyse)
nb_pays_conserves = len(base_analyse_prete)
nb_pays_exclus = nb_pays_fao_initial - nb_pays_conserves

taux_conservation_pays = nb_pays_conserves / nb_pays_fao_initial * 100

population_initiale = base_analyse["population_2017"].sum()
population_conservee = base_analyse_prete["population_2017"].sum()
taux_couverture_population = population_conservee / population_initiale * 100

print("Nombre de pays FAO initiaux :", nb_pays_fao_initial)
print("Nombre de pays conservés :", nb_pays_conserves)
print("Nombre de pays exclus :", nb_pays_exclus)
print("Taux de conservation des pays :", round(taux_conservation_pays, 2), "%")
print("Taux de couverture de la population :", round(taux_couverture_population, 2), "%")

Nombre de pays FAO initiaux : 170
Nombre de pays conservés : 163
Nombre de pays exclus : 7
Taux de conservation des pays : 95.88 %
Taux de couverture de la population : 99.08 %


In [129]:
# Sélection de la base d'analyse avant intégration de la variable bio

colonnes_base_analyse_finale = [
    "code_iso3",
    "pays",
    "region_banque_mondiale",
    "niveau_revenu",

    "population_2017",
    "dispo_interieure_volaille",
    "nourriture_volaille",
    "dispo_kg_hab_volaille",
    "dispo_proteines_volaille",
    "production_volaille",
    "importations_volaille",
    "exportations_volaille",
    "dependance_importation_volaille",
    "importations_kg_hab_volaille",
    "production_kg_hab_volaille",
    "solde_import_export_volaille",

    "pib_habitant_ppa",
    "population_urbaine_pct",
    "stabilite_politique",
    "qualite_reglementaire",
    "performance_logistique",
    "droit_douane_poulet_moyen_pct",
    "doing_business_score",
    "score_cout_import_moyen",

    "nb_valeurs_manquantes_pestel_initial",
    "pib_habitant_ppa_imputee",
    "performance_logistique_imputee",
    "droit_douane_poulet_moyen_pct_imputee"
]

base_analyse_finale = base_analyse_prete[
    colonnes_base_analyse_finale
].copy()

print("Dimensions de la base avant variable bio :", base_analyse_finale.shape)
print("Nombre de pays :", base_analyse_finale["pays"].nunique())
print("Nombre de codes ISO3 uniques :", base_analyse_finale["code_iso3"].nunique())
print("Nombre de doublons sur code_iso3 :", base_analyse_finale["code_iso3"].duplicated().sum())
print("Nombre total de valeurs manquantes :", base_analyse_finale.isna().sum().sum())

base_analyse_finale.head()


Dimensions de la base avant variable bio : (163, 28)
Nombre de pays : 163
Nombre de codes ISO3 uniques : 163
Nombre de doublons sur code_iso3 : 0
Nombre total de valeurs manquantes : 0


,code_iso3,pays,region_banque_mondiale,niveau_revenu,population_2017,dispo_interieure_volaille,nourriture_volaille,dispo_kg_hab_volaille,dispo_proteines_volaille,production_volaille,importations_volaille,exportations_volaille,dependance_importation_volaille,importations_kg_hab_volaille,production_kg_hab_volaille,solde_import_export_volaille,pib_habitant_ppa,population_urbaine_pct,stabilite_politique,qualite_reglementaire,performance_logistique,droit_douane_poulet_moyen_pct,doing_business_score,score_cout_import_moyen,nb_valeurs_manquantes_pestel_initial,pib_habitant_ppa_imputee,performance_logistique_imputee,droit_douane_poulet_moyen_pct_imputee
0,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,36296113.0,57.0,55.0,1.53,0.54,28.0,29.0,0.0,50.877193,0.80,0.77,29.0,2335.795862,24.835282,-2.613931,-1.416448,1.95,10.000000,38.93563,18.750000,1,0,0,1
1,ZAF,Afrique du Sud,Sub-Saharan Africa,Upper middle income,57009756.0,2118.0,2035.0,35.69,14.11,1667.0,514.0,63.0,23.567171,9.02,29.24,451.0,13738.438585,63.331108,-0.382832,0.229679,3.38,16.827381,65.40686,66.973215,0,0,0,0
2,ALB,Albanie,Europe & Central Asia,Upper middle income,2884169.0,47.0,47.0,16.36,6.26,13.0,38.0,0.0,74.509804,13.18,4.51,38.0,14110.683242,55.980276,0.163830,0.110524,2.66,10.000000,64.16093,96.064880,0,0,0,0
3,DZA,Algérie,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,41389189.0,277.0,264.0,6.38,1.97,275.0,2.0,0.0,0.722022,0.05,6.64,2.0,13493.560749,71.322791,-0.929505,-0.945914,2.45,30.000001,46.10981,54.412040,0,0,0,0
4,DEU,Allemagne,Europe & Central Asia,High income,82658409.0,1739.0,1609.0,19.47,7.96,1514.0,842.0,646.0,35.738540,10.19,18.32,196.0,54110.253665,80.984254,0.702224,1.694667,4.20,3.200000,79.55430,100.000000,0,0,0,0


## Intégration d’une variable complémentaire liée à l’agriculture biologique

Une variable complémentaire liée à l'agriculture biologique est testée afin d'évaluer la maturité du marché ou de l'écosystème bio dans chaque pays.

La variable retenue est la part des terres arables consacrées à l'agriculture biologique. Elle ne mesure pas directement la consommation de produits bio, mais elle constitue une proxy objective de la maturité de l'écosystème bio.

L'objectif est de vérifier si cette variable présente une couverture suffisante pour être intégrée à la base d'analyse finale.


In [130]:
# Test de récupération d'une variable liée à l'agriculture biologique

url_bio = "https://ourworldindata.org/grapher/share-of-arable-land-which-is-organic.csv"

headers_bio = {
    "User-Agent": "Mozilla/5.0"}

response_bio = requests.get(
    url_bio,
    headers=headers_bio,
    timeout=60)

print("Statut de la requête :", response_bio.status_code)

response_bio.raise_for_status()

bio_brut = pd.read_csv(BytesIO(response_bio.content))

print("Dimensions :", bio_brut.shape)

print("\nColonnes disponibles :")
display(bio_brut.columns)

bio_brut.head()

Statut de la requête : 200
Dimensions : (3675, 4)

Colonnes disponibles :


Index(['Entity', 'Code', 'Year', 'Share of arable land which is organic'], dtype='object')

,Entity,Code,Year,Share of arable land which is organic
0,Afghanistan,AFG,2007,0.0
1,Afghanistan,AFG,2008,0.0
2,Afghanistan,AFG,2009,0.0
3,Afghanistan,AFG,2010,0.0
4,Afghanistan,AFG,2011,0.0


In [131]:
# Test de couverture de la variable bio sur la base d'analyse prête

bio_2017 = bio_brut[
    bio_brut["Year"] == 2017
][[
    "Code",
    "Entity",
    "Share of arable land which is organic"]].copy()

bio_2017 = bio_2017.rename(columns={
    "Code": "code_iso3",
    "Entity": "pays_bio",
    "Share of arable land which is organic": "part_terres_arables_bio_pct"})

bio_2017 = bio_2017[
    bio_2017["code_iso3"].notna()].copy()

base_bio_test = base_analyse_prete.merge(
    bio_2017[[
        "code_iso3",
        "part_terres_arables_bio_pct"]],
    on="code_iso3",
    how="left")

nb_pays_base_finale = len(base_bio_test)
nb_pays_avec_bio = base_bio_test["part_terres_arables_bio_pct"].notna().sum()
nb_pays_sans_bio = base_bio_test["part_terres_arables_bio_pct"].isna().sum()
taux_couverture_bio = nb_pays_avec_bio / nb_pays_base_finale * 100

print("Nombre de pays dans la base d'analyse :", nb_pays_base_finale)
print("Nombre de pays avec donnée bio :", nb_pays_avec_bio)
print("Nombre de pays sans donnée bio :", nb_pays_sans_bio)
print("Taux de couverture bio :", round(taux_couverture_bio, 2), "%")

print("\nStatistiques descriptives de la variable bio :")
display(base_bio_test["part_terres_arables_bio_pct"].describe())

print("\nPays sans donnée bio :")
display(
    base_bio_test[
        base_bio_test["part_terres_arables_bio_pct"].isna()
    ][[
        "code_iso3",
        "pays",
        "region_banque_mondiale",
        "niveau_revenu"]])

Nombre de pays dans la base d'analyse : 163
Nombre de pays avec donnée bio : 138
Nombre de pays sans donnée bio : 25
Taux de couverture bio : 84.66 %

Statistiques descriptives de la variable bio :


count    138.000000
mean       3.087754
std        5.852581
min        0.000000
25%        0.132500
50%        0.605000
75%        3.382500
max       44.680000
Name: part_terres_arables_bio_pct, dtype: float64


Pays sans donnée bio :


,code_iso3,pays,region_banque_mondiale,niveau_revenu
5,AGO,Angola,Sub-Saharan Africa,Lower middle income
6,ATG,Antigua-et-Barbuda,Latin America & Caribbean,High income
15,BRB,Barbade,Latin America & Caribbean,High income
20,BWA,Botswana,Sub-Saharan Africa,Upper middle income
24,BLR,Bélarus,Europe & Central Asia,Upper middle income
31,HKG,Chine - RAS de Hong-Kong,East Asia & Pacific,High income
35,COG,Congo,Sub-Saharan Africa,Lower middle income
40,DJI,Djibouti,"Middle East, North Africa, Afghanistan & Pakistan",Lower middle income
50,GAB,Gabon,Sub-Saharan Africa,Upper middle income
51,GMB,Gambie,Sub-Saharan Africa,Low income


In [132]:
# Intégration de la variable bio dans la base finale

base_analyse_finale_bio = base_bio_test.copy()

# Indicateur de valeur imputée
base_analyse_finale_bio["part_terres_arables_bio_pct_imputee"] = (
    base_analyse_finale_bio["part_terres_arables_bio_pct"].isna().astype(int))

# Imputation par médiane des pays comparables : région + niveau de revenu
base_analyse_finale_bio["part_terres_arables_bio_pct"] = (
    base_analyse_finale_bio
    .groupby([
        "region_banque_mondiale",
        "niveau_revenu"
    ])["part_terres_arables_bio_pct"]
    .transform(lambda serie: serie.fillna(serie.median())))

# Si certaines valeurs restent manquantes, imputation par médiane du niveau de revenu
base_analyse_finale_bio["part_terres_arables_bio_pct"] = (
    base_analyse_finale_bio
    .groupby("niveau_revenu")["part_terres_arables_bio_pct"]
    .transform(lambda serie: serie.fillna(serie.median())))

# Dernier recours : médiane globale
base_analyse_finale_bio["part_terres_arables_bio_pct"] = (
    base_analyse_finale_bio["part_terres_arables_bio_pct"]
    .fillna(base_analyse_finale_bio["part_terres_arables_bio_pct"].median()))

print("Dimensions de la base finale avec variable bio :", base_analyse_finale_bio.shape)

print("\nNombre de valeurs bio imputées :")
print(base_analyse_finale_bio["part_terres_arables_bio_pct_imputee"].sum())

print("\nValeurs manquantes restantes :")
display(base_analyse_finale_bio.isna().sum())

print("\nStatistiques descriptives après imputation :")
display(base_analyse_finale_bio["part_terres_arables_bio_pct"].describe())

base_analyse_finale_bio.head()

Dimensions de la base finale avec variable bio : (163, 30)

Nombre de valeurs bio imputées :
25

Valeurs manquantes restantes :


code_iso3                                0
pays                                     0
population_2017                          0
dispo_interieure_volaille                0
nourriture_volaille                      0
dispo_kg_hab_volaille                    0
dispo_proteines_volaille                 0
production_volaille                      0
importations_volaille                    0
exportations_volaille                    0
dependance_importation_volaille          0
importations_kg_hab_volaille             0
production_kg_hab_volaille               0
solde_import_export_volaille             0
region_banque_mondiale                   0
niveau_revenu                            0
pib_habitant_ppa                         0
population_urbaine_pct                   0
stabilite_politique                      0
qualite_reglementaire                    0
performance_logistique                   0
droit_douane_poulet_moyen_pct            0
doing_business_score                     0
score_cout_


Statistiques descriptives après imputation :


count    163.000000
mean       2.694939
std        5.479520
min        0.000000
25%        0.125000
50%        0.480000
75%        2.710000
max       44.680000
Name: part_terres_arables_bio_pct, dtype: float64

,code_iso3,pays,population_2017,dispo_interieure_volaille,nourriture_volaille,dispo_kg_hab_volaille,dispo_proteines_volaille,production_volaille,importations_volaille,exportations_volaille,dependance_importation_volaille,importations_kg_hab_volaille,production_kg_hab_volaille,solde_import_export_volaille,region_banque_mondiale,niveau_revenu,pib_habitant_ppa,population_urbaine_pct,stabilite_politique,qualite_reglementaire,performance_logistique,droit_douane_poulet_moyen_pct,doing_business_score,score_cout_import_moyen,nb_valeurs_manquantes_pestel_initial,pib_habitant_ppa_imputee,performance_logistique_imputee,droit_douane_poulet_moyen_pct_imputee,part_terres_arables_bio_pct,part_terres_arables_bio_pct_imputee
0,AFG,Afghanistan,36296113.0,57.0,55.0,1.53,0.54,28.0,29.0,0.0,50.877193,0.80,0.77,29.0,"Middle East, North Africa, Afghanistan & Pakistan",Low income,2335.795862,24.835282,-2.613931,-1.416448,1.95,10.000000,38.93563,18.750000,1,0,0,1,0.00,0
1,ZAF,Afrique du Sud,57009756.0,2118.0,2035.0,35.69,14.11,1667.0,514.0,63.0,23.567171,9.02,29.24,451.0,Sub-Saharan Africa,Upper middle income,13738.438585,63.331108,-0.382832,0.229679,3.38,16.827381,65.40686,66.973215,0,0,0,0,0.04,0
2,ALB,Albanie,2884169.0,47.0,47.0,16.36,6.26,13.0,38.0,0.0,74.509804,13.18,4.51,38.0,Europe & Central Asia,Upper middle income,14110.683242,55.980276,0.163830,0.110524,2.66,10.000000,64.16093,96.064880,0,0,0,0,0.05,0
3,DZA,Algérie,41389189.0,277.0,264.0,6.38,1.97,275.0,2.0,0.0,0.722022,0.05,6.64,2.0,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,13493.560749,71.322791,-0.929505,-0.945914,2.45,30.000001,46.10981,54.412040,0,0,0,0,0.00,0
4,DEU,Allemagne,82658409.0,1739.0,1609.0,19.47,7.96,1514.0,842.0,646.0,35.738540,10.19,18.32,196.0,Europe & Central Asia,High income,54110.253665,80.984254,0.702224,1.694667,4.20,3.200000,79.55430,100.000000,0,0,0,0,6.82,0


### Traitement de la variable liée à l'agriculture biologique

La variable `part_terres_arables_bio_pct` mesure la part des terres arables consacrées à l'agriculture biologique. Elle est utilisée comme proxy de la maturité de l'écosystème bio dans chaque pays.

Cette variable présentait une couverture de 84,66 % sur la base finale, avec 138 pays renseignés sur 163. Elle a donc été intégrée à l'analyse.

Les valeurs manquantes restantes ont été imputées selon la même logique que les autres variables complémentaires :

1. imputation par la médiane des pays appartenant à la même région Banque mondiale et au même niveau de revenu ;
2. si nécessaire, imputation par la médiane des pays appartenant au même niveau de revenu ;
3. en dernier recours, imputation par la médiane globale.

La médiane a été privilégiée afin de limiter l'influence des valeurs extrêmes.

Une variable indicatrice, `part_terres_arables_bio_pct_imputee`, permet d'identifier les pays pour lesquels la valeur bio a été imputée.

La variable `part_terres_arables_bio_pct` ne mesure pas directement la consommation de produits biologiques. Elle est utilisée comme une proxy de la maturité de l'écosystème bio dans le pays, en reflétant la part des terres arables consacrée à l'agriculture biologique ou en conversion.

Comme l’entreprise vend du poulet bio, j’ai intégré une variable permettant d’approcher la maturité de l’écosystème biologique dans chaque pays. Ce n’est pas une mesure directe de consommation, mais elle donne un signal complémentaire sur la sensibilité potentielle au bio.

In [133]:
# Export final de la base d'analyse avec la variable bio

base_analyse_finale = base_analyse_finale_bio.copy()

base_analyse_finale.to_csv("base_analyse_finale.csv", index=False)

print("Dimensions de la base finale :", base_analyse_finale.shape)
print("Nombre de pays :", base_analyse_finale["pays"].nunique())
print("Nombre de codes ISO3 uniques :", base_analyse_finale["code_iso3"].nunique())
print("Nombre de doublons sur code_iso3 :", base_analyse_finale["code_iso3"].duplicated().sum())
print("Nombre total de valeurs manquantes :", base_analyse_finale.isna().sum().sum())
print("Fichier exporté :", Path("base_analyse_finale.csv").exists())

assert base_analyse_finale["pays"].nunique() == 163
assert base_analyse_finale["code_iso3"].duplicated().sum() == 0
assert base_analyse_finale.isna().sum().sum() == 0
assert Path("base_analyse_finale.csv").exists()

print("\nBase finale validée et exportée.")

base_analyse_finale.head()


Dimensions de la base finale : (163, 30)
Nombre de pays : 163
Nombre de codes ISO3 uniques : 163
Nombre de doublons sur code_iso3 : 0
Nombre total de valeurs manquantes : 0
Fichier exporté : True

Base finale validée et exportée.


,code_iso3,pays,population_2017,dispo_interieure_volaille,nourriture_volaille,dispo_kg_hab_volaille,dispo_proteines_volaille,production_volaille,importations_volaille,exportations_volaille,dependance_importation_volaille,importations_kg_hab_volaille,production_kg_hab_volaille,solde_import_export_volaille,region_banque_mondiale,niveau_revenu,pib_habitant_ppa,population_urbaine_pct,stabilite_politique,qualite_reglementaire,performance_logistique,droit_douane_poulet_moyen_pct,doing_business_score,score_cout_import_moyen,nb_valeurs_manquantes_pestel_initial,pib_habitant_ppa_imputee,performance_logistique_imputee,droit_douane_poulet_moyen_pct_imputee,part_terres_arables_bio_pct,part_terres_arables_bio_pct_imputee
0,AFG,Afghanistan,36296113.0,57.0,55.0,1.53,0.54,28.0,29.0,0.0,50.877193,0.80,0.77,29.0,"Middle East, North Africa, Afghanistan & Pakistan",Low income,2335.795862,24.835282,-2.613931,-1.416448,1.95,10.000000,38.93563,18.750000,1,0,0,1,0.00,0
1,ZAF,Afrique du Sud,57009756.0,2118.0,2035.0,35.69,14.11,1667.0,514.0,63.0,23.567171,9.02,29.24,451.0,Sub-Saharan Africa,Upper middle income,13738.438585,63.331108,-0.382832,0.229679,3.38,16.827381,65.40686,66.973215,0,0,0,0,0.04,0
2,ALB,Albanie,2884169.0,47.0,47.0,16.36,6.26,13.0,38.0,0.0,74.509804,13.18,4.51,38.0,Europe & Central Asia,Upper middle income,14110.683242,55.980276,0.163830,0.110524,2.66,10.000000,64.16093,96.064880,0,0,0,0,0.05,0
3,DZA,Algérie,41389189.0,277.0,264.0,6.38,1.97,275.0,2.0,0.0,0.722022,0.05,6.64,2.0,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,13493.560749,71.322791,-0.929505,-0.945914,2.45,30.000001,46.10981,54.412040,0,0,0,0,0.00,0
4,DEU,Allemagne,82658409.0,1739.0,1609.0,19.47,7.96,1514.0,842.0,646.0,35.738540,10.19,18.32,196.0,Europe & Central Asia,High income,54110.253665,80.984254,0.702224,1.694667,4.20,3.200000,79.55430,100.000000,0,0,0,0,6.82,0


## Conclusion du notebook de préparation des données

Ce notebook a permis de construire une base d'analyse consolidée à partir des données FAO et d'indicateurs complémentaires externes.

La base finale contient 163 pays et regroupe des variables relatives :

- au marché de la volaille ;
- à la population ;
- au pouvoir d'achat ;
- à l'urbanisation ;
- à la stabilité politique ;
- à la qualité réglementaire ;
- à la performance logistique ;
- aux droits de douane appliqués au poulet ;
- à l'environnement des affaires ;
- aux coûts d'importation ;
- à la maturité de l'écosystème bio.

Les pays présentant un nombre trop important de valeurs PESTEL manquantes ont été exclus. Les valeurs manquantes restantes ont été imputées par médiane de pays comparables, avec conservation d'indicateurs permettant d'identifier les valeurs imputées.

La base finale ne contient plus de valeurs manquantes et peut être utilisée pour les analyses exploratoires, l'ACP et les méthodes de classification.


## Sources de données

Les données utilisées dans ce notebook proviennent de plusieurs sources open data sélectionnées pour leur pertinence par rapport à l’objectif du projet : identifier des pays présentant un potentiel pour l’exportation de poulet bio.

| Source | Utilisation dans le projet | Lien |
|---|---|---|
| FAOSTAT — Food and Agriculture Organization | Données de population, disponibilité alimentaire, production, importations et exportations de viande de volaille | [FAOSTAT](https://www.fao.org/faostat/en/) |
| Banque mondiale — Country API | Référentiel pays, codes ISO3, régions et niveaux de revenu | [World Bank Country API](https://api.worldbank.org/v2/country?format=json) |
| Banque mondiale — Indicators API | PIB par habitant PPA, population urbaine et performance logistique | [World Bank Indicators API](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392-about-the-indicators-api-documentation) |
| Worldwide Governance Indicators — WGI | Stabilité politique et qualité réglementaire | [Worldwide Governance Indicators](https://www.worldbank.org/en/publication/worldwide-governance-indicators) |
| WITS — World Integrated Trade Solution | Droits de douane appliqués aux produits de viande de poulet | [WITS](https://wits.worldbank.org/) |
| WITS API | Récupération des produits et données tarifaires par code HS | [WITS API](https://wits.worldbank.org/witsapiintro.aspx?lang=en) |
| Doing Business — Banque mondiale | Score Doing Business et scores liés aux coûts d’importation | [Doing Business — Trading across Borders](https://archive.doingbusiness.org/en/data/exploretopics/trading-across-borders) |
| Our World in Data | Part des terres arables consacrées à l’agriculture biologique | [Share of arable land which is organic](https://ourworldindata.org/grapher/share-of-arable-land-which-is-organic) |

Le site Données Mondiales était mentionné dans l’énoncé comme source possible, mais il n’a pas été utilisé dans cette analyse. Les sources retenues ont été privilégiées pour leur couverture pays, leur structure exploitable en Python et leur cohérence avec l’analyse PESTEL.